In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn import functional as F

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt

In [ ]:
# Set the random seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
# Set the device, cuda or cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device}")

# 0. Introduction to Large Language Models

A large language model, in a nutshell, is doing just one thing: given a string of "text", it guesses what the next piece of "text" will be, and then repeats this process in a loop until it meets a stopping condition (for example, the model emits the agreed-upon end-of-sequence token EOS, or it has already generated up to the maximum allowed length). And when we call it "large", we mean that its model size is "large" and its training data is "large".

For a language model, the given "text" is called a token, which can be the words or characters we use in everyday life, or other things that constitute the elements of language. That is, given a string of tokens, the model predicts the next token based on the information it is given (next-token prediction). Written out as a formula:

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

Here, on the conditioning side, $t_{1}, t_{2}, ..., t_{k-1}$ denote the first $k-1$ tokens and $\theta$ denotes the model, while on the other side $t_{k}$ denotes the predicted $k$-th token; as for $p(\cdot)$, it is the probability the model assigns to this $k$-th token—so the so-called "guessing" is really a matter of "guessing one token according to probability", over and over again. (We will come back to this perspective later when we discuss tokenization.)

What a large language model does is really just this simple, and when the model size and the training data are scaled up in tandem, the model's performance improves predictably with scale—this is the so-called scaling laws; and over the course of learning from this massive amount of data, the model gradually comes to acquire all sorts of remarkable abilities.

In this tutorial, we learn the key techniques of LLM training and inference by training a "small but complete" model (a toy model that still has all the essential parts).

# 1. Data Preparation

## 1.1. Download the Shakespeare Dataset

We use the works of Shakespeare as training data, so that the model learns to write in Shakespeare's style.

In [ ]:
if not os.path.exists("./data/shakespeare.txt"):
    os.makedirs("./data", exist_ok=True)  # wget -O won't auto-create the parent directory, so create ./data first
    !wget -O ./data/shakespeare.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
    #the dataset is downloaded to ./data, named shakespeare.txt
else:
    print("file has been downloaded")

## 1.2. Inspect the dataset

In [ ]:
with open("./data/shakespeare.txt", "r", encoding = "utf-8") as f:
    text = f.read()

# Number of characters in the dataset
print("text len: {}".format(len(text)))

In [ ]:
# View the first 500 characters of the dataset
print(text[:500])

## 1.3. Tokenization

***Converting text into minimal units of meaning***

For a human, the semantic hierarchy of Shakespeare's works, or of any English text, looks like this: the 26 letters make up words, words plus punctuation form sentences, sentences combine into paragraphs, and paragraphs are stitched together into an article. We can see that each semantic level has its corresponding minimal unit of meaning, which from the bottom up are, in order: letter $\rightarrow$ word $\rightarrow$ sentence $\rightarrow$ paragraph $\rightarrow$ article. A model, however, is dumber than a human: it cannot automatically build such a hierarchy, nor does it know what the minimal unit of meaning is. So we have to manually give the model such a prior, namely the minimal unit of meaning at the lowest semantic level, which we call a token. We then let the model, by learning from large amounts of data, spontaneously build a higher-level, more abstract understanding of text, and even integrate it all into a coherent whole, giving rise to "emergence".

For English text, a token does not necessarily have to start from a character: a token can be a character, a subword (a sequence of characters), a word, or a phrase (a combination of words). Different ways of choosing tokens each have their pros and cons: if we use characters as tokens, the "vocabulary" is very small (there are fewer than a hundred distinct characters in English text), but the gap between simple characters and complex semantic expression is too large, which makes learning extremely hard for the model; at the other extreme, suppose we use phrases as tokens: a single token can then express complex semantics, but the vocabulary becomes especially large (there are far too many different combinations of words), and learning may likewise be very hard for the model. So tokenization is an operation that requires a trade-off. There are many ready-made algorithms for determining tokens, for example [BPE](https://github.com/openai/tiktoken), an algorithm that extracts subwords (character sequences) from large amounts of text according to their frequency of occurrence.

At its core, a large language model is doing just one thing: next-token prediction. That is, given a sequence of tokens as input, the model predicts what the next token is. Formally:

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

Given the first $k-1$ tokens, the model $\theta$ gives the probability of the $k$-th token. On the surface this is merely a probabilistic statistical model, but once we feed it all the text humanity has produced since antiquity (that is, all of human knowledge) for training, the model can produce a sensible $k$-th token only if it has come to understand the intricate workings of the vast world embodied in the countless combinations of those first $k-1$ tokens.

### 1.3.1. Characters as tokens

Here we start with the simplest and most direct approach: treating characters as tokens. Suppose the Shakespeare text contains N distinct characters in total (for example, English characters or punctuation marks); we map each character to a discrete integer (0 to N-1, N in total). Then next-token prediction becomes the following task: given a sequence of tokens (represented by a corresponding sequence of integers between 0 and N-1), output a vector of length N, whose i-th dimension represents the probability that the token to be predicted is the i-th character.

In [ ]:
# Print all the characters
chars = sorted(list(set(text)))
print("There are {} chars, which are {}".format(len(chars), chars))

In [ ]:
# each character corresponds to a discrete integer
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# test encode and decode
print(encode("hi there"))
print(decode(encode("hi there")))

### 1.3.2. (optional) The BPE Algorithm

We can also use the more advanced [BPE](https://github.com/openai/tiktoken) (Byte Pair Encoding) algorithm for tokenization. It is precisely the kind of algorithm mentioned above that "extracts subwords from large amounts of text according to their frequency of occurrence."

**The principle in one sentence**: Starting from the most basic units (characters/bytes), repeatedly merge "the most frequent adjacent token pair in the current corpus" into a new token, and repeat this until the vocabulary grows to a preset size. In this way, high-frequency subwords (common words, roots, and affixes, such as `ing` and `tion`) are gradually merged into single tokens, while rare strings are left as smaller fragments — thus striking a balance between the two extremes of "vocabulary size" and "sequence length". This echoes the same trade-off discussed above: the finer the token granularity, the smaller the vocabulary, but the longer the sequence and the larger the span from characters to semantics, making it harder for the model to learn; coarser granularity is the opposite.

**The procedure**: Let the base symbol set (all characters or bytes) be $V_0$, and write every word in the corpus as a sequence of base symbols. Then repeat the following two steps $M$ times ($M$ is the preset number of merges, a hyperparameter):

$$
(x^{*}, y^{*}) = \arg\max_{(x, y)}\ \mathrm{freq}(x, y)
$$

$$
V_{t+1} = V_t \cup \{x^{*} y^{*}\}, \qquad t = 0, 1, \dots, M-1
$$

That is: on the **current corpus**, count the frequency $\mathrm{freq}(x, y)$ of every adjacent token pair $(x, y)$, take the most frequent pair $(x^{*}, y^{*})$, merge it into a new token $x^{*}y^{*}$, and add it to the vocabulary, then replace **all** occurrences of that adjacent pair in the corpus with this new token (if there is a tie in frequency, pick one by some convention); each merge rewrites the corpus once, so the frequencies for the next round must be recounted on the **new corpus**. The final vocabulary size is $|V| = |V_0| + M$. When encoding new text, you only need to greedily merge step by step according to the learned merge rules **and the order in which they were learned**; when decoding, simply concatenate the subwords end to end to recover the original.

**A small example**: Suppose the corpus contains the following words and word frequencies `{hug:10, pug:5, pun:12, bun:4, hugs:5}`, with the base vocabulary `{b, g, h, n, p, s, u}`. In each round, take the most frequent adjacent pair at that moment and merge it:

| Round | Most frequent adjacent pair | Frequency | New token |
| :--: | :--: | :--: | :--: |
| 1 | (u, g) | 20 | `ug` |
| 2 | (u, n) | 16 | `un` |
| 3 | (h, ug) | 15 | `hug` |

Thus "hug" is encoded as a single token `hug`, while "hugs" is `hug`+`s`; and a new word "bug" that never appeared in the corpus can still be split into the two tokens `b`+`ug` with the help of the learned rules — this is exactly why BPE can comfortably handle out-of-vocabulary (OOV) words.

**Modern practice**: The GPT family uses **byte-level BPE**, i.e., it builds merges on raw **bytes** (rather than characters), and the base vocabulary is always the 256 bytes. Therefore, any text (various languages, emoji, and even garbled text) is never OOV, and since spaces, newlines, and even any character itself are byte-level base symbols, the original text can be reassembled byte by byte, **completely losslessly**. The `tiktoken` (`cl100k_base` / `o200k_base`) that we will use below is precisely an efficient implementation of this kind of byte-level BPE.

For a more detailed derivation, see [Introduction to Natural Language Processing: Byte Pair Encoding (BPE) and Natural Language Toolkit (NLTK)](https://medium.com/@RobuRishabh/introduction-to-natural-language-processing-byte-pair-encoding-bpe-and-natural-language-toolkit-414c83f85b71#:~:text=Byte%20Pair%20Encoding%20(BPE)%20is,smaller%20units%20called%20subword%20tokens.&text=Handling%20Out%2Dof%2DVocabulary%20Words,words%20not%20seen%20during%20training.).

In [ ]:
!pip3 install -U tiktoken

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")#("o200k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

For detailed usage, see the [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb)

#### 1.3.2.1. A closer look: how GPT's byte-level BPE actually works

Naive BPE merges directly on **characters**, which runs into two problems: the base vocabulary has to cover every possible character (Unicode has over a hundred thousand code points), and the moment it hits a character not seen during training (rare characters, new emoji) it still goes OOV. Starting from GPT-2, BPE instead operates on **bytes**, cleanly sidestepping both. Concretely, there are four steps:

1. **Pre-tokenization**: first use a regex to split the text into "word chunks" (roughly splitting on whitespace / punctuation / digits, and folding a leading space into the word, so `" the"` and `"the"` are different chunks). **BPE merges happen only inside a chunk, never across chunks**, so whitespace and punctuation naturally become boundaries, preventing `"dog."` and `"dog"` from being learned as the same token. (cl100k / o200k use an updated regex that handles contractions, runs of digits, etc. more finely.)
2. **Convert to bytes (UTF-8)**: encode each chunk into a **byte stream** with UTF-8. Then, whatever the language, symbol, or emoji, it all becomes a string of bytes in $0\sim255$ — **the base vocabulary is always exactly these 256 bytes**, and there will never again be an "unseen character".
3. **Run BPE merges on the bytes**: apply the "merge the most frequent adjacent pair by frequency" procedure described earlier to the byte sequence, learning a merge table. The final vocabulary = 256 base bytes + the learned merges (cl100k_base has about 100k in total). Note: a common English word may be merged into a single token, whereas a rare emoji may dutifully split into several byte tokens — **the cost is more tokens, but it never fails**.
4. **Encode / decode**: when encoding, greedily merge each chunk into token ids following the priority order of the merge table; when decoding, turn each token id back into its corresponding **byte sequence**, concatenate them all, and UTF-8-decode back into text. Because everything ultimately bottoms out in bytes and all 256 bytes are in the vocabulary, **the round trip is completely lossless** — emoji and multi-byte Chinese characters can be reassembled byte by byte.

> A small detail: GPT-2's tokenizer additionally maps the 256 bytes one-to-one onto printable Unicode characters (reversibly), purely to make the vocabulary file readable and to keep whitespace / control characters from causing trouble — this is why you see spellings like `Ġthe` (where `Ġ` stands for a space) in the GPT-2 vocabulary. `tiktoken` (cl100k), by contrast, works directly on the raw bytes, and `enc._mergeable_ranks` stores exactly the `byte -> rank` mapping.

In [ ]:
# byte-level BPE intuitive demo: a string containing accent marks, Chinese, and emoji
s = "héllo 世界 🤖"
ids = enc.encode(s)
print("Original     :", repr(s))
print("token ids:", ids)

# Each token essentially corresponds to a sequence of bytes
print("Bytes per token:")
for i in ids:
    print("   {:>6}  ->  {}".format(i, enc.decode_single_token_bytes(i)))

# Everything ultimately comes down to bytes, so the round-trip is completely lossless (emoji and multi-byte Chinese characters are no exception)
print("Lossless round-trip :", enc.decode(ids) == s)

# The base vocabulary is simply the 256 bytes — this is precisely the foundation of "no text is ever OOV"
base_bytes = sum(1 for b in range(256) if bytes([b]) in enc._mergeable_ranks)
print("Single-byte tokens in the vocabulary:", base_bytes, "(= all 256 bytes)")
print("cl100k_base total vocabulary size n_vocab:", enc.n_vocab)

In [ ]:
# ==========
use_bpe = False # character-level by default (the "65 characters / N=65 / 65 probability values" references below are all based on this);
               # set it to True to switch to BPE — but note that vocab_size will jump from 65 to roughly 100,000, and the earlier explanations about 65 no longer apply,
               # and at this toy scale (n_embed=32, block_size=16) the vocabulary is far larger than the data, so both training and generation quality degrade noticeably
# ==========

if use_bpe:
    encode = enc.encode
    decode = enc.decode

    print(encode("hi there"))
    print(decode(encode("hi there")))
    
    vocab_size_bpe = len(enc._mergeable_ranks)

## 1.4. Converting the Shakespeare dataset into tokens

In [ ]:
# Encode the Shakespeare dataset from above
data = encode(text)
len_data = len(data)
print("len of data: {}".format(len_data))

Split into training and test sets

In [ ]:
split_point = int(0.9 * len(data))
train_set = data[:split_point]
test_set = data[split_point:]
print(f"len of train_set: {len(train_set)}")
print(f"len of test_set: {len(test_set)}")

# 2. Model hyperparameters

In [ ]:
# These values were chosen arbitrarily and are not optimal; feel free to modify them

batch_size = 128 # training batch size
block_size = 16 # maximum number of tokens fed into the model
if use_bpe:
    vocab_size = vocab_size_bpe
else:
    vocab_size = len(chars) # vocabulary size, which is the number of unique characters in this case
n_embed = 32 # dimension of the embedding layer. The integer representing a token is fed into the model, which maps this integer to a vector of dimension n_embed.
n_layers = 16 # number of multihead attention block layers in the model
head_size = 16 # dimension of each head in the model's multihead attention block
num_heads = 8 # number of heads in the model's multihead attention block (note that head_size*num_heads=128 is not equal to n_embed=32: q/k/v are first projected to 128 dimensions to compute attention, then projected back to n_embed by fc; the two are decoupled in this implementation, which differs from the common convention of d_model=num_heads*head_size)

learning_rate = 1e-4 # learning rate
print_interval = 200 # interval for printing training information; training information is printed every print_interval steps
num_epoch = 10 # number of training epochs

# 3. The model's dataset loader

Define the input and output for each training step

e.g. (assuming characters are used as tokens)

- Model input: `e?\n\nDUKE OF AUME` prediction target: `?\n\nDUKE OF AUMER`

- Model input: `rike?\n\nSecond Mu` prediction target: `ike?\n\nSecond Mur`

In [ ]:
# Define Dataset and Datasetloader
class MyDataset(Dataset):
    def __init__(self, textdata, block_size):
        self.textdata = textdata
        self.block_size = block_size
    
    def __len__(self):
        return len(self.textdata) - self.block_size  # must ensure idx+1+block_size <= len, so the last valid idx is len-block_size-1, hence the number of samples is len-block_size
    
    def __getitem__(self, idx):
        return {
            "input": torch.tensor(self.textdata[idx:idx+self.block_size], dtype=torch.long),
            "target": torch.tensor(self.textdata[idx+1:idx+1+self.block_size], dtype=torch.long),
        }

train_dataset = MyDataset(train_set, block_size)
train_datasetloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers = 4)

test_dataset = MyDataset(test_set, block_size)
test_datasetloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers = 4)

In [ ]:
# Get a batch of inputs and prediction targets
example_batch = next(iter(train_datasetloader))

# tensor shape of the model inputs and prediction targets = (batch_size, block_size)
print("input shape (batch_size, block_size): {}\ntarget shape (batch_size, block_size): {}\n".format(example_batch["input"].shape, example_batch["target"].shape))

# Examples of the model inputs and prediction targets
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(decode(list(example_batch["input"][0].numpy()))), repr(decode(list(example_batch["target"][0].numpy())))))

# What it looks like without decoding
print("without decoding\nexample input: {}\nexample target: {}".format(list(example_batch["input"][0].numpy()), list(example_batch["target"][0].numpy())))

In [ ]:
# Test whether the datasetloader works correctly

# for i, batch in enumerate(train_datasetloader):
#     if i % 100 == 0:
#         print("{} | {}".format(i, len(train_datasetloader)), end="\r")
# print("{} | {}".format(i, len(train_datasetloader)))
# print("there is no error")

# 4. Embedding

***Map tokens to high-dimensional vectors that represent both semantics (individual) and positional relationships (mutual)***

## 4.1. Semantic embedding

In the Tokenization step, suppose we define a token as a character and represent each token with an integer from 0 to N-1 (the Shakespeare dataset contains 65 distinct characters, so N is 65). To enable the model to understand and process these discrete integers, we map each integer to a vector of length `n_embed`, where every dimension of the vector is a continuous value. Concretely, here we have 65 characters and `n_embed`=32, so we have an $E \in \mathbb{R}^{65 \times 32}$, where the $i$-th row of $E$ represents the $i$-th token. The point of this is that the vector of length `n_embed` corresponding to the $i$-th token can be viewed as the model's abstract understanding of the $i$-th token, and since this vector is differentiable, it can be trained and updated.

In [ ]:
# Define a model with only a single embedding layer
class EmbeddingTest(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        return self.token_embed_tab(x)

embed_test_model = EmbeddingTest(
    vocab_size,
    n_embed,
)

In [ ]:
# Get a batch of inputs and prediction targets
example_batch = next(iter(train_datasetloader))

# tensor shape of the model input and prediction target = (batch_size, block_size)
print("input shape (batch_size, block_size): {}\ntarget shape (batch_size, block_size): {}\n".format(example_batch["input"].shape, example_batch["target"].shape))

# An example of the model input and prediction target
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(decode(list(example_batch["input"][0].numpy()))), repr(decode(list(example_batch["target"][0].numpy())))))

# What it looks like without decoding
print("without decoding\nexample input: {}\nexample target: {}".format(list(example_batch["input"][0].numpy()), list(example_batch["target"][0].numpy())))

In [ ]:
# embedding
embed = embed_test_model(example_batch["input"])
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))

## 4.2. Position Embedding

***Adding positional information to tokens***

The embedding approach above helps the model understand the meaning of each token itself (its features are abstracted by the model into a vector of length `n_embed`), but in this setting the model cannot learn the "mutual" relationships between tokens, because such an embedding contains no information that characterizes a token's "position". We therefore introduce positional encoding (Position Embedding).

There are many kinds of positional encoding, which can be roughly divided into two categories: **absolute positional encoding** and **relative positional encoding**. For an introduction to the various encoding schemes, see [Transformer Positional Encoding That Makes Researchers Rack Their Brains (in Chinese)](https://spaces.ac.cn/archives/8130).
Here we present the three most representative positional encodings.

### 4.2.1. Absolute Position - Learned Position Embedding

The simplest and most direct encoding scheme: in the same vein as the semantic embedding in 4.1 — we build another **learnable** position table $P \in \mathbb{R}^{16 \times 32}$ ($16$=block_size, $32$=n_embed), where the $i$-th row is the encoding of the $i$-th position (randomly initialized and updated during training). During the forward pass, we look up the corresponding rows by indices $0, 1, \dots, T-1$ and add them to the token embedding. Here $i$ is merely an index for "which position", not a value that is directly treated as a parameter and learned.

In [ ]:
# Add a simple learned position embedding
class EmbeddingTestWithSelfLearningPosEmbed(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        block_size
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_tab = nn.Embedding(block_size, n_embed)

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        tok_embed = self.token_embed_tab(x)
        pos_embed = self.pos_embed_tab(torch.arange(T, device = x.get_device() if x.get_device() >= 0 else "cpu"))
        x = tok_embed + pos_embed
        return x, tok_embed, pos_embed

embed_test_model_w_pos = EmbeddingTestWithSelfLearningPosEmbed(
    vocab_size,
    n_embed,
    block_size
)

In [ ]:
embed, tok_embed, pos_embed = embed_test_model_w_pos(example_batch["input"])

# text
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# Example of model input and prediction target
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# shapes
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))
print("shape of pos_embed (block_size, n_embed) = {}".format(pos_embed.shape))

### 4.2.2. Absolute Position - Sinusoidal Position Embedding

From [Attention Is All You Need](https://arxiv.org/abs/1706.03762)

Unlike learned positional encoding, sinusoidal positional encoding is **fixed and requires no learning**: using a set of sine and cosine functions of different frequencies, it directly computes a vector of length $d$ (i.e., `n_embed`) for each absolute position $pos$, which is then added to the embedding of the corresponding token.

Let $pos \in \{0, 1, \dots, L-1\}$ be the position of a token in the sequence ($L$ is `block_size`), $d$ the embedding dimension, and $i \in \{0, 1, \dots, d/2-1\}$ the index of a dimension pair; then:

$$
\begin{aligned}
PE_{(pos,\, 2i)}   &= \sin\left(\frac{pos}{10000^{2i/d}}\right) \\
PE_{(pos,\, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d}}\right)
\end{aligned}
$$

That is, the $d$ dimensions are grouped two by two, and the $i$-th group shares a single frequency $\omega_i = 1/10000^{2i/d}$: the even dimension (dimension $2i$) takes $\sin(\omega_i\, pos)$, and the odd dimension (dimension $2i+1$) takes $\cos(\omega_i\, pos)$ — this corresponds one-to-one with `pe[:, 0::2] = sin` and `pe[:, 1::2] = cos` in the code below. Afterwards, `x = tok_embed + pos_embed`.

**Intuition**:

- **Multi-scale "dials"**: the frequency $\omega_i$ decays geometrically as $i$ increases, with the wavelength stretching from $2\pi$ all the way out to about $2\pi \cdot 10000$. The low dimensions behave like fast-moving hands and the high dimensions like slow-moving hands; different dimensions oscillate at different speeds, and taken together they give each position an almost unique "fingerprint" (much like using a row of pointers spinning at different speeds to represent an integer).
- **Implicit relative position**: the pairing of sine and cosine is no accident — for any fixed offset $k$, $PE_{pos+k}$ can be written as a **linear transformation** of $PE_{pos}$ (essentially a block-wise rotation: the two dimensions of the $i$-th group each rotate by $\omega_i k$, and every group's rotation angle varies only linearly with $k$, independent of $pos$). So once the model learns this linear relationship, it can "read off" the relative distance between tokens from the absolute encoding.
- **Extrapolatable**: because it is a deterministic formula rather than lookup-table learning, in theory it can directly compute encodings for longer positions never seen during training (although in practice the extrapolation effect is limited).

For further reading, see this article [The Road to Upgrading Transformers: 1. Tracing the Origins of Sinusoidal Positional Encoding](https://spaces.ac.cn/archives/8231) (in Chinese)

In [ ]:
class EmbeddingTestWithSinPosEmbed(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        block_size,
        base=10000
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        
        self.max_sequence_length = block_size
        self.d_model = n_embed
        self.base = base
        self.register_buffer("pos_embed", self.get_pos_embed())  # register as a buffer so it moves along with the module on .to(device)

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        tok_embed = self.token_embed_tab(x)
        x = tok_embed + self.pos_embed
        return x, tok_embed, self.pos_embed
        
    def get_pos_embed(self):
        pe = torch.zeros(self.max_sequence_length, self.d_model, dtype=torch.float)  # size: (max_sequence_length, d_model) = (block_size, n_embed), the number of rows is the "number of positions" block_size, don't confuse it with the token embedding's vocab_size
        exp_value = 2*torch.arange(self.d_model // 2, dtype=torch.float) / self.d_model
        alpha = 1 / (self.base ** exp_value)  # size: (d_model/2)
        out = torch.arange(self.max_sequence_length, dtype=torch.float)[:, None] @ alpha[None, :]  # size: (max_sequence_length, d_model/2)
        embedding_sin = torch.sin(out)
        embedding_cos = torch.cos(out)
        pe[:, 0::2] = embedding_sin  # even dimensions (2i) set to sin
        pe[:, 1::2] = embedding_cos  # odd dimensions (2i+1) set to cos
        return pe

embed_test_model_w_pos = EmbeddingTestWithSinPosEmbed(
    vocab_size,
    n_embed,
    block_size
)

In [ ]:
embed, tok_embed, pos_embed = embed_test_model_w_pos(example_batch["input"])

# Text
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# Example of model input and prediction target
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# Shapes
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))
print("shape of pos_embed (block_size, n_embed) = {}".format(pos_embed.shape))

### 4.2.3. Relative Position-RoPE

RoPE (Rotary Position Embedding) takes a different approach: instead of **adding** positional information to the embedding, it **rotates** the vector by an angle according to its position; moreover, it acts on the query / key that enter attention (for intuitive clarity, the demo code in this section applies it directly to the token embedding).

We group the $d$-dimensional vector pairwise into $d/2$ two-dimensional planes, and assign the $i$-th group a frequency $\theta_i = 1/10000^{2i/d}$ ($i = 0, \dots, d/2-1$, the same geometric decay as sinusoidal encoding). For a vector $x$ at position $m$, the components of its $i$-th group $(x_{2i},\, x_{2i+1})$ are rotated by the angle $m\theta_i$:

$$
\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix}
=
\begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix}
\begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}
$$

It is more compact to write this with complex numbers (which is exactly what the code does with `view_as_complex` + `torch.polar`): letting $z_i = x_{2i} + \mathrm{i}\, x_{2i+1}$, the rotation is simply multiplication by a unit complex number:

$$
\tilde{z}_i = z_i \cdot e^{\mathrm{i}\, m \theta_i}
$$

**intuition (why an "absolute" rotation can nonetheless express "relative position")**:

Let $R_m$ denote the operation of "performing the above block-wise rotation according to position $m$". The core of attention is the inner product of query and key; applying RoPE to $q$ at position $m$ and $k$ at position $n$ respectively, their inner product is

$$
\langle R_m\, q,\ R_n\, k \rangle = q^{\top} R_m^{\top} R_n\, k = q^{\top} R_{n-m}\, k
$$

because the rotation matrix satisfies $R_m^{\top} R_n = R_{n-m}$. In other words: **although what is applied to each token is an "absolute-position" rotation, what attention actually perceives is only the relative distance $n-m$ between the two**. This is precisely the elegance of RoPE — it is implemented in an absolute-position manner (simple, and requiring no extra pairwise positional-bias matrix), yet reaps the effect of relative position.

A vivid analogy: think of each token's content as an arrow on a plane; the later the position, the more the arrow gets rotated. The difference between the angles through which the two arrows have each been rotated (i.e. $(n-m)\theta_i$) depends only on how far apart they are, and is independent of their individual absolute positions; different dimension groups rotate at different speeds, which serves to distinguish "close together" from "far apart".

It is precisely this clever knack of "being absolute in form yet relative in effect", together with its simple implementation and natural fit with the attention mechanism, that has made RoPE adopted by almost all modern open-source large models, such as Llama 3, Qwen, DeepSeek, Gemma, and Mistral. Note, however, that the original RoPE often has limited effectiveness when extrapolated directly beyond the training length; in practice it is commonly combined with techniques such as NTK-aware, Position Interpolation / YaRN to extend the context length — which is exactly the subject that several articles in the blog series cited below are devoted to.

For details, see [Transformer Upgrade Path: 2. A Rotary Position Embedding That Draws on the Strengths of Many](https://spaces.ac.cn/archives/8265/comment-page-1) (in Chinese)

In [ ]:
class EmbeddingTestWithRoPE(nn.Module):
    def __init__(
        self,
        n_embed,
        block_size,
        base=10000
    ):
        super().__init__()        
        self.max_sequence_length = block_size
        self.d_model = n_embed
        self.base = base
        self.register_buffer("freqs_cis", self.init_pos_embed())  # register as a buffer; it will be moved along when .to(device) is called

    def forward(self, x):
        batch_size, sequence_length, d_model = x.shape
        assert self.d_model == d_model and self.max_sequence_length == sequence_length
        
        h = x.float().reshape(*x.shape[:-1], -1, 2) # --> shape: (batch_size, self.max_sequence_length, self.d_model//2, 2)
        h = torch.view_as_complex(h) # --> shape: (batch_size, self.max_sequence_length, self.d_model//2)
        
        # rotation operation
        # let the d-th dimension of the t-th token in the b-th sample along the batch dimension of h, i.e. (b, t, d), be m + ni; then multiply it by the corresponding complex number in freqs_cis, i.e. the complex number at position (t, d). Assuming the rotation angle is a, then (m + ni) * (cos(a)+sin(a)i)
        # after h is multiplied by the complex numbers at the corresponding positions in freqs_cis and converted back to real numbers, shape: (batch_size, self.max_sequence_length, self.d_model//2) --> (batch_size, self.max_sequence_length, self.d_model)
        h = torch.view_as_real(h * self.freqs_cis).flatten(2) # --> shape: (batch_size, self.max_sequence_length, self.d_model)
        h = h.type_as(x)
        
        return h, x
        
    def init_pos_embed(self):
        # after grouping the elements of the dim dimension into pairs, compute the rotation angle for each dim element in each pair
        freqs = 1.0 / (self.base ** (torch.arange(0, self.d_model, 2)[:(self.d_model//2)].float() / self.d_model)) # shape: (self.d_model//2)
        # generate the indices along the token sequence dimension
        t = torch.arange(self.max_sequence_length, device=freqs.device) # shape: (self.max_sequence_length)
        # outer product of freqs and t; (i,j) is the rotation angle of the j-th dimension of the i-th token
        freqs = torch.outer(t, freqs).float() # shape: (self.max_sequence_length, self.d_model//2)
        # convert the rotation angles in freqs to complex form; if the rotation angle at position (i,j) in freqs is a, then after conversion the (i,j) position of freqs_cis becomes cos(a)+sin(a)i
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs) # shape: (self.max_sequence_length, self.d_model//2)
        return freqs_cis

embed_test_model_rope = EmbeddingTestWithRoPE(
    n_embed,
    block_size
)

In [ ]:
_, tok_embed, _ = embed_test_model_w_pos(example_batch["input"])

# Text
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# Example of model input and prediction target
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# token embed
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))

# Add RoPE
x_add_rope, _ = embed_test_model_rope(tok_embed)
print("shape of x (batch_size, block_size, n_embed) adding rope = {}".format(x_add_rope.shape))

# 5. Attention in Transformer

Many articles have explained the principles of attention in Transformer in detail; I recommend the following two:

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) ("the illustrations are excellent")

- [Attention? Attention!](https://lilianweng.github.io/posts/2018-06-24-attention/) (written by an expert at OpenAI; it also reviews the history of attention, as there were many different flavors of attention before Transformer)

Still, let me explain attention in Transformer here from a code perspective.

First, why do we need attention? Next-token prediction in a language model ultimately boils down to the following:

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

Conditioned on model parameters $\theta$, and given tokens $t_{1}, t_{2}, ..., t_{k-1}$, predict the probability that the next token is $t_{k}$ (in the Shakespeare dataset example above, $t_{k}$ can be any one of the 65 characters, so we obtain 65 probability values, and the highest value indicates the character most likely to be the next token). Within this framework, predicting the next character requires referring to the preceding characters. The question, then, is: exactly how should this referencing be done? Earlier architectures such as RNNs and LSTMs devised various "referencing" tricks, and attention was proposed with the same goal in mind — but in practice, attention has proven to work better.

When discussing attention, the most crucial piece is this formula (which I'm sure everyone has seen countless times, but I'll put it here anyway):

$$
Attention(Q, K ,V) = softmax(\frac{QK^{T}}{\sqrt{d}})V
$$

For now, let's set aside what this formula is doing and return to our code. After passing through the embedding, each token is abstracted into a vector of length `n_embed`. In other words, this batch contains `batch_size` training samples; for each training sample, the number of input tokens is `block_size`; and each token is represented by a vector of length `n_embed`.

In [ ]:
x = embed
print("shape of x (batch_size, block_size, n_embed) = {}".format(x.shape))

# Here x represents the model's intermediate-layer features; we apply the following Attention operations to this x...

## 5.1. Self-Attention

In the Transformer's Attention operation, the Key and Value may come from a different source than the Query (the variant where they come from different sources becomes Cross-Attention), whereas when all three share the same source it is Self-Attention. The language model we are going to build here uses Self-Attention throughout.

In [ ]:
# Obtain Query, Key, Value through Linear layers
class QueryKeyValue(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
    ):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
    
    def forward(self, x):
        return self.query(x), self.key(x), self.value(x)

get_query_key_value = QueryKeyValue(n_embed, head_size=n_embed) # Here the dimension (head_size) of Query, Key, Value is still n_embed

After passing the input `x` through three separate Linear layers, we obtain the query, key, and value (i.e., all three are derived from `x`), namely `q, k ,v`

In [ ]:
q, k ,v = get_query_key_value(x)

print("shape of query (batch_size, block_size, head_size): {}".format(q.shape))
print("shape of key (batch_size, block_size, head_size): {}".format(k.shape))
print("shape of value (batch_size, block_size, head_size): {}".format(v.shape))

Here the batch size > 1; we take any one of the samples for illustration and reduce the batch size to 1 — the case with batch size > 1 works in exactly the same way.

In [ ]:
q, k, v = q[:1], k[:1], v[:1] # shape: (1, block_size, head_size)
sample_input_text = input_text_list[0]
sample_target_text = target_text_list[0]
print(repr("Now, the corresponding input text is {} and target text is {}".format(repr(sample_input_text), repr(sample_target_text))))

First, let's compute the $\frac{QK^{T}}{\sqrt{d}}$ step, where $d$ is the last dimension of the Q and K vectors (in this demo, since the previous cell set head_size=n_embed, we have $d$=n_embed=32; in a real model, $d$=head_size).

In [ ]:
weights = q @ k.transpose(-2, -1) * q.shape[-1]**-0.5 # scale by the actual last dimension of Q/K (in this demo, since head_size=n_embed is set above, this is equivalent to dividing by sqrt(n_embed))
print("shape: {}".format(weights.shape)) # shape: (batchsize, num of tokens, num of tokens)

Let me explain why we need the $\frac{1}{\sqrt{d}}$ factor. It is there to control the variance. If we do not divide by $\sqrt{d}$, the magnitude of $QK^{T}$ grows as the dimension $d$ increases, and after softmax the probability becomes overly concentrated on a single token (close to one-hot), making the gradient extremely small and training difficult; dividing by $\sqrt{d}$ pulls the variance back to $O(1)$, so that softmax does not saturate. Below, we use random numbers to intuitively observe how the variance changes:

In [ ]:
# Use random numbers as an example
k_rand = torch.randn(batch_size, block_size, head_size)
q_rand = torch.randn(batch_size, block_size, head_size)
weights_rand = q_rand @ k_rand.transpose(-2, -1) * head_size**-0.5
weights_rand_without_frac_d = q_rand @ k_rand.transpose(-2, -1)

print("var of k_rand: {}".format(k_rand.var()))
print("var of q_rand: {}".format(q_rand.var()))
print("var of weights_rand: {}".format(weights_rand.var()))
print("when 1/sqrt(d) is not applied, var of weights_rand_without_frac_d: {}".format(weights_rand_without_frac_d.var()))

In [ ]:
# For our q, k
print("var of k: {}".format(k.var()))
print("var of q: {}".format(q.var()))
print("var of weights: {}".format(weights.var()))
print("without 1/sqrt(d), var of q @ k.transpose(-2, -1): {}".format((q @ k.transpose(-2, -1)).var()))

So what does the expression $\frac{QK^{T}}{\sqrt{d}}$ itself actually mean?

Here, the step `weights = q @ k.transpose(-2, -1) * head_size**-0.5` performs the shape transformation `(1, block_size, head_size) @ (1, head_size, block_size) --> (1, block_size, block_size)`. In the resulting `weights`, the entry at row $i$, column $j$ represents the importance of the $j$-th token to the $i$-th token.

We now plot the result of `weights`.

In [ ]:
attn_tick_labels = [decode([token]) for token in encode(sample_input_text)] #[*sample_input_text]
for i in range(len(attn_tick_labels)):
    if attn_tick_labels[i] == "\n":
        attn_tick_labels[i] = "\\n"
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("weights")
plt.show()

However, for the task of next-token prediction, this kind of attention has a problem: the $i$-th token can only see itself and the tokens before it (e.g., the $1, 2, ..., i$-th tokens), but it should not see the tokens that come after it (the `tril` used below includes the diagonal, precisely to preserve each token's attention to "itself"). To address this, the causal attention operation is introduced, namely:

In [ ]:
# a lower-triangular matrix of shape = (block_size, block_size)
tril_matrix = torch.tril(torch.ones(block_size, block_size))
print(tril_matrix)

For the `weights` obtained above, set the positions where the corresponding `tril_matrix` is 0 to `-inf`. In other words, for row $i$ of `weights` (which represents the $i$-th token), all tokens with index $>i$ have no influence on the $i$-th token (their importance is negative infinity).

In [ ]:
weights = weights.masked_fill(tril_matrix == 0, float("-inf"))
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("causal weights")
plt.show()

Then apply the `softmax` operation, which yields $softmax(\frac{QK^{T}}{\sqrt{d}})$

In [ ]:
weights = F.softmax(weights, dim=-1)
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("softmax of causal weights")
plt.show()

Thus, the $i$-th row here represents the importance of all tokens to the $i$-th token; of course, all tokens after the $i$-th have an importance value of 0

In [ ]:
ith = torch.randint(low=0, high=weights.shape[1], size=(1,))[0].item()
wei_val_ith = weights[0, ith].detach().cpu().numpy()
print("the {}th row of weights: {}\n\nthe sum of the weights of the {}th row is {}\n".format(ith+1, wei_val_ith, ith+1, weights[0, ith].sum()))
print("which means, for the {}th token '{}'".format(ith+1, attn_tick_labels[ith]))
for j, (cur_wei_val, label) in enumerate(zip(wei_val_ith, attn_tick_labels)):
    print("the importance of {}th token '{}' to {}th token '{}' is {}".format(j+1, label, ith+1, attn_tick_labels[ith], cur_wei_val))

For $Attention(Q, K ,V) = softmax(\frac{QK^{T}}{\sqrt{d}})V$, we have already obtained $softmax(\frac{QK^{T}}{\sqrt{d}})$ and know what it means (the `weights` mentioned above, which characterize the degree of importance between tokens). So what does it mean to multiply $softmax(\frac{QK^{T}}{\sqrt{d}})$ by the matrix $V$?

As described above, the $i$-th row of $softmax(\frac{QK^{T}}{\sqrt{d}})$ characterizes the degree of importance of all the other tokens to the $i$-th token, while the $i$-th row of $V$ represents the features of the $i$-th token.

Then multiplying $softmax(\frac{QK^{T}}{\sqrt{d}})$ by $V$ can be viewed as multiplying each row of $softmax(\frac{QK^{T}}{\sqrt{d}})$ by $V$.

Multiplying the $i$-th row of $softmax(\frac{QK^{T}}{\sqrt{d}})$ by $V$ means assigning weights to the features of all the tokens in $V$ and then summing them, which yields the features of the $i$-th token after it has undergone the attention operation with the other tokens, i.e.,

$$
\text{the } i\text{-th row of } softmax(\frac{QK^{T}}{\sqrt{d}}): softmax(\frac{QK^{T}}{\sqrt{d}})_{i} = [a_{1 \rightarrow i}, a_{2 \rightarrow i}, ..., a_{b \rightarrow i}]
$$

$$
V = [f_{1}, f_{2}, ..., f_{b}]^{T}
$$

$$
softmax(\frac{QK^{T}}{\sqrt{d}})_{i}V = a_{1 \rightarrow i}f_{1} + a_{2 \rightarrow i}f_{2} + a_{b \rightarrow i}f_{b}
$$

where $b$ is `block_size`; $a_{j \rightarrow i}$ is a number between 0 and 1 that characterizes the degree of importance of the $j$-th token to the $i$-th token (as described above); $f_{j} \in \mathbb{R}^{1 \times d}$ is the feature of the $j$-th token, where $d$ is `head_size`.

In [ ]:
# This gives us the output
out = weights @ v # (1, block_size, block_size) @ (1, block_size, head_size) --> (1, block_size, head_size)
print("shape of out: {}".format(out.shape))

To summarize, let's organize the code.

In [ ]:
class SimpleSelfAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.fc = nn.Linear(head_size, n_embed) # map the feature dimension from head_size back to n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))
        
    def forward(self, x):
        # x.shape = (batch_size, block_size, n_embed)
        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size), K shape: (batch_size, block_size, head_size), V shape: (batch_size, block_size, head_size)
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)
        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, block_size, head_size) @ (batch_size, head_size, block_size) --> (batch_size, block_size, block_size)
        w = w.masked_fill(self.mask[:w.shape[-2],:w.shape[-1]]==0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, block_size, block_size) @ (batch_size, block_size, head_size) --> (batch_size, block_size, head_size)
        out = self.fc(out) # --> (batch_size, block_size, n_embed)
        return out

simple_self_attn = SimpleSelfAttn(
    n_embed = n_embed,
    head_size = head_size,
    block_size_max = block_size,
)

In [ ]:
print("shape of input of attention module: {}".format(x.shape))
print("shape of output of attention module: {}".format(simple_self_attn(x).shape))

## 5.2. Multi-head Attention

On top of the self-attention described above, we introduce a technique called multi-head attention. The name sounds fancy, but in essence it just extends `q, k, v` from a single head to multiple heads: first we widen the projection to `head_size * num_heads` dimensions, then split it into `num_heads` smaller heads each of `head_size` dimensions, i.e. `(batch_size, block_size, head_size*num_heads) --> (batch_size, block_size, num_heads, head_size)` (note that `head_size * num_heads` need not equal `n_embed`; in the end `fc` projects it back to `n_embed`). The extra `num_heads` dimension lets the model learn a separate set of attention features in each of several subspaces. In addition, multi-head attention is computed in parallel, so it is computationally efficient.

In [ ]:
class MultiHeadAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads, # number of heads in multi-head attention
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads

        self.key = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.fc = nn.Linear(head_size*num_heads, n_embed) # map the feature dimension from head_size back to n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    # function to split into heads
    # ====================
    def split_heads(self, h):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * self.num_heads
        h = h.reshape(batch_size, block_size, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h
    # ====================

    def forward(self, x):
        # x.shape = (batch_size, block_size, n_embed)

        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size*num_heads),
        # K shape: (batch_size, block_size, head_size*num_heads),
        # V shape: (batch_size, block_size, head_size*num_heads)

        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)

        # split out the num_heads dimension from q, k, v
        # ====================
        q = self.split_heads(q) # (batch_size, block_size, head_size*num_heads) --> (batch_size, num_heads, block_size, head_size)
        k = self.split_heads(k)
        v = self.split_heads(v)
        # ====================

        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, num_heads, block_size, head_size) @ (batch_size, num_heads, head_size, block_size) --> (batch_size, num_heads, block_size, block_size)
        w = w.masked_fill(self.mask[:w.shape[-2],:w.shape[-1]]==0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, num_heads, block_size, block_size) @ (batch_size, num_heads, block_size, head_size) --> (batch_size, num_heads, block_size, head_size)

        # reshape out to merge the features from the different heads
        # ====================
        out = out.permute(0,2,1,3) # (batch_size, num_heads, block_size, head_size) --> (batch_size, block_size, num_heads, head_size)
        out = out.reshape(batch_size, seq_length, self.num_heads*self.head_size) # (batch_size, block_size, num_heads, head_size) --> (batch_size, block_size, num_heads*head_size)
        # ====================

        out = self.fc(out) # (batch_size, block_size, num_heads*head_size) --> (batch_size, block_size, n_embed)
        return out

multi_head_attn = MultiHeadAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
)

In [ ]:
print("shape of input of multi-head attention module: {}".format(x.shape))
print("shape of output of multi-head attention module: {}".format(multi_head_attn(x).shape))

## 5.3. KV cache: Why Inference Can Be Faster

For the two attention variants above we did not distinguish between training and inference, but training only needs to "predict the next token", whereas inference has to predict over and over in a loop—that is, feeding each prediction back as a condition in order to predict the next one. If caching is not enabled during inference, then every time we generate a token we feed the **entire sequence** back into the model and run a full forward pass again—so the key / value of all those earlier tokens are **recomputed over and over**, a sheer waste.

Hence the **KV cache** technique, which "trades space for time". The idea is simple: because the attention is causal, the representation of a past token never changes when new tokens are appended afterward. Let's walk through a very simple mathematical derivation. Suppose we currently have $2$ tokens, so $Q = [q_{1}; q_{2}], K = [k_{1}; k_{2}], V = [v_{1}; v_{2}]$; then our result also has two entries. The first entry can be abbreviated as $softmax_{row-wise}(q_{1}k_{1}^{T})v_{1}$, which depends only on the first token, and the second entry can be abbreviated as $softmax_{row-wise}(q_{2}k_{1}^{T})v_{1} + softmax_{row-wise}(q_{2}k_{2}^{T})v_{2}$, which depends only on the first and second tokens. Here $softmax_{row-wise}$ is a shorthand for the $softmax$ operation; we only need to pay attention to which $q, k$ it involves. Now, computing the next token from the first two, the result should be $softmax_{row-wise}(q_{3}k_{1}^{T})v_{1} + softmax_{row-wise}(q_{3}k_{2}^{T})v_{2} + softmax_{row-wise}(q_{3}k_{3}^{T})v_{3}$. As you can see, $k_{1}, v_{1}, k_{2}, v_{2}$ are still the same as before; we only need to newly compute $q_{3}, k_{3}, v_{3}$.

The **KV cache** method is now obvious: **cache** the $K, V$ of past tokens; at each step compute $q, k, v$ only for **the one newly arrived token**, append the new $k, v$ to the cache, and then simply let the new query attend once over "all of the $K/V$ in the cache". The per-step attention thus drops from "recomputing over the entire sequence of $O(T)$ tokens" to "computing just 1 new token and reading the cache once", greatly accelerating autoregressive generation—at the cost of keeping this cache resident in GPU memory, with a size proportional to the number of KV heads.

**Why cache only K and V, and not Q as well?** Because the whole point of a cache is to "store something now so it can be reused later": the key / value of a past token are looked up again and again by the new query at **every step** thereafter, so they are worth storing; whereas each query is used only once, at its own step. At each step the attention matrix gains only **one row** (the new query), yet needs **all columns** (all keys / values): the columns are reused over and over, so they are cached; a row is discarded right after use, which is why there is no q-cache.

Below we use a minimal single-head causal self-attention example to see clearly how the KV cache mechanism works, and to verify that it is element-for-element identical to "recomputing the whole sequence":

In [ ]:
# KV cache principle demo: single-head causal self-attention as the example,
# comparing "recompute the whole sequence at every step" (the use_cache=False reference implementation) with "cache K/V and compute only the new token each step" (KV cache)
torch.manual_seed(0)
d = head_size
# just fabricate a set of projection weights
Wq = torch.randn(d, d)
Wk = torch.randn(d, d)
Wv = torch.randn(d, d)

def causal_attn(X):                # no cache: run causal self-attention once over the whole X
    q, k, v = X @ Wq, X @ Wk, X @ Wv
    w = q @ k.transpose(-2, -1) * d ** -0.5
    T = X.shape[0]
    w = w.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))  # causal mask
    return F.softmax(w, dim=-1) @ v

T = 6
X = torch.randn(T, d)              # assume these are the features of T tokens arriving one at a time

# Approach A (KV cache disabled): for each new token, recompute the whole [0..t] segment and take the output at the last position
out_recompute = [causal_attn(X[:t + 1])[-1] for t in range(T)]

# Approach B (KV cache): compute q/k/v only for "the single newly arrived token", append the new k/v to the cache,
# then have the new query attend to "all K/V in the cache" (naturally causal, not even a mask needed)
K_cache, V_cache, out_cache = [], [], []
for t in range(T):
    x_t = X[t]
    q_t, k_t, v_t = x_t @ Wq, x_t @ Wk, x_t @ Wv
    K_cache.append(k_t); V_cache.append(v_t)       # historical k/v are never recomputed, only the new one is appended
    K = torch.stack(K_cache); V = torch.stack(V_cache)
    w = (q_t @ K.transpose(-2, -1)) * d ** -0.5
    out_cache.append(F.softmax(w, dim=-1) @ V)

# the two are element-wise equal: KV cache merely skips redundant computation, the result is unchanged
print("KV cache matches full recomputation:", torch.allclose(torch.stack(out_recompute), torch.stack(out_cache), atol=1e-5))
print("Full recomputation runs {} token forward passes in total, KV cache runs only {}".format(sum(range(1, T + 1)), T))

Add a **KV cache** to the `MultiHeadAttn` above

In [ ]:
class MultiHeadAttnKVCache(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads, # define the number of heads in multi-head attention
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        
        self.key = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.fc = nn.Linear(head_size*num_heads, n_embed) # map the feature dimension from head_size back to n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    # function to split heads
    # ====================
    def split_heads(self, h):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * self.num_heads
        h = h.reshape(batch_size, block_size, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h
    # ====================

    def forward(self, x, past_kv = None, use_cache = False):
        # x.shape = (batch_size, block_size, n_embed)
        
        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size*num_heads),
        # K shape: (batch_size, block_size, head_size*num_heads),
        # V shape: (batch_size, block_size, head_size*num_heads)
        
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)

        # split out the num_heads dimension from q, k, v
        # ====================
        q = self.split_heads(q) # (batch_size, block_size, head_size*num_heads) --> (batch_size, num_heads, block_size, head_size)
        k_new = self.split_heads(k)
        v_new = self.split_heads(v)
        # ====================
        
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k_new], dim = 2) # concatenate along the time dimension: history + new token
            v = torch.cat([past_v, v_new], dim = 2)
        else:
            k, v = k_new, v_new
            
        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, num_heads, block_size, head_size) @ (batch_size, num_heads, head_size, block_size) --> (batch_size, num_heads, block_size, block_size)
        # the causal mask must be offset by the cached length: the i-th new query (absolute position past_len+i) can only attend to key positions ≤ past_len+i
        T_kv = k.size(2)
        past_len = T_kv - seq_length  # cached history length (= starting absolute position of the new query)
        w = w.masked_fill(self.mask[past_len:past_len + seq_length, :T_kv] == 0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, num_heads, block_size, block_size) @ (batch_size, num_heads, block_size, head_size) --> (batch_size, num_heads, block_size, head_size)

        # reshape out to merge the features from different heads
        # ====================
        out = out.permute(0,2,1,3) # (batch_size, num_heads, block_size, head_size) --> (batch_size, block_size, num_heads, head_size)
        out = out.reshape(batch_size, seq_length, self.num_heads*self.head_size) # (batch_size, block_size, num_heads, head_size) --> (batch_size, block_size, num_heads*head_size)
        # ====================
        
        out = self.fc(out) # (batch_size, block_size, num_heads*head_size) --> (batch_size, block_size, n_embed)
        return (out, (k,v)) if use_cache else out

multi_head_attn_kv_cache = MultiHeadAttnKVCache(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
)

In [ ]:
# Verify MultiHeadAttnKVCache: the result of per-token incremental decoding (with KV cache) is element-wise identical to the "full recompute"
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # features of the T tokens assumed to arrive step by step

with torch.no_grad():
    full = multi_head_attn_kv_cache(seq)                              # Approach A: one-shot forward pass over the whole sequence (reference)
    past, steps = None, []                                           # Approach B: per-token incremental decoding
    for t in range(T):
        out_t, past = multi_head_attn_kv_cache(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)
    incr = torch.cat(steps, dim=1)

ok = torch.allclose(full, incr, atol=1e-5)
print("incremental decode == full recompute:", ok, "| max error {:.1e}".format((full - incr).abs().max().item()))
assert ok, "KV-cache incremental decoding should be element-wise identical to the full recompute"
past_k, past_v = past
print("cached K/V shape (B, num_heads, T, head_size):", tuple(past_k.shape))

## 5.4. Group Query Attention

Multi-head attention is great, but it has a pain point during **autoregressive inference**: every time a new token is generated, its query must attend to the key and value of all preceding tokens. To avoid redundant computation, in practice the key and value of past tokens are cached — this is the **KV cache**. In MHA, each query head has its own dedicated key/value head, so the size of the KV cache is proportional to the number of heads — when the sequence is long and the model is large, the GPU memory and bandwidth needed to read and write this cache often become the bottleneck for inference speed.

The idea behind **Grouped-Query Attention (GQA)** is straightforward: let multiple query heads **share** the same key/value head. The $H$ query heads are split into $G$ groups ($G$ divides $H$), and the $H/G$ query heads within the same group share a single key/value head; this reduces the number of key/value heads from $H$ to $G$, and the KV cache shrinks accordingly to $G/H$ of its original size.

It neatly connects the two extremes:

| Method | # key/value heads | Description |
| :-- | :--: | :-- |
| **MHA** (Multi-Head Attention) | $G = H$ | Each query head has its own dedicated KV; best expressiveness, largest KV cache |
| **GQA** (Grouped-Query Attention) | $1 < G < H$ | A middle ground: the KV cache drops to $G/H$, with quality close to MHA |
| **MQA** (Multi-Query Attention) | $G = 1$ | All query heads share one set of KV; smallest KV cache, largest quality loss |

Denoting the query head index as $h \in \{0, 1, \dots, H-1\}$, the key/value head it uses is

$$
g(h) = \left\lfloor \frac{h}{H/G} \right\rfloor \in \{0, 1, \dots, G-1\}
$$

The implementation differs from MHA in only one place: key and value are projected into only $G$ heads (rather than $H$), and before computing attention each key/value head is **replicated** $H/G$ times to align with the query heads in its group; the subsequent $\mathrm{softmax}\!\left(QK^\top/\sqrt{d}\right)V$ is exactly the same as in MHA. GQA trades a tiny quality cost for a KV cache several times smaller, and is widely adopted by many modern open-source models such as Llama 3, Qwen, Gemma, and Mistral (DeepSeek-V3 goes one step further; see MLA below).

In addition, the `GroupQueryAttn.forward` below also supports **incremental decoding with the KV cache** (the `past_kv` / `use_cache` parameters; when disabled by default, its behavior is identical to the original), and the code cell further down demonstrates it and verifies that it exactly matches recomputing the entire sequence.

In [ ]:
class GroupQueryAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads,      # number of query heads H
        num_kv_heads,   # number of key/value heads G, must divide num_heads; =num_heads reduces to MHA, =1 reduces to MQA
        block_size_max,
    ):
        super().__init__()
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_queries_per_kv = num_heads // num_kv_heads # how many query heads in a group share each kv head (= the number of copies to replicate, H/G; note this is the "queries per group", as distinguished from the number of groups G=num_kv_heads in the main text)

        # Note: query projects out num_heads heads, whereas key/value project out only num_kv_heads heads (this is exactly where the smaller KV cache comes from)
        self.query = nn.Linear(n_embed, head_size * num_heads,    bias = False)
        self.key   = nn.Linear(n_embed, head_size * num_kv_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size * num_kv_heads, bias = False)
        self.fc = nn.Linear(head_size * num_heads, n_embed) # map the feature dimension from head_size*num_heads back to n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def split_heads(self, h, num_heads):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * num_heads
        h = h.reshape(batch_size, block_size, num_heads, self.head_size)
        return h.permute(0, 2, 1, 3) # (batch_size, num_heads, block_size, head_size)

    def forward(self, x, past_kv = None, use_cache = False):
        # x.shape = (batch_size, T_q, n_embed)
        #   · training / one-shot forward pass: x is the whole sequence, past_kv=None, use_cache=False (behaves exactly like the original, returns only out)
        #   · inference-time incremental decoding: x feeds in only the "newly arrived token" (T_q is usually =1), past_kv passes in the cache from the previous step, use_cache=True
        batch_size, T_q, n_embed = x.size()
        assert n_embed == self.n_embed

        q     = self.split_heads(self.query(x), self.num_heads)    # (B, H, T_q, head_size)
        k_new = self.split_heads(self.key(x),   self.num_kv_heads) # (B, G, T_q, head_size)  ← only G kv heads
        v_new = self.split_heads(self.value(x), self.num_kv_heads) # (B, G, T_q, head_size)

        # ===== KV cache: append this step's newly computed k/v onto the end of the historical cache =====
        # the cache always stores G kv heads (not H)—this is exactly the GPU memory GQA saves over MHA; historical k/v are no longer recomputed
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k_new], dim = 2) # concatenate along the time dimension: history + new token
            v = torch.cat([past_v, v_new], dim = 2)
        else:
            k, v = k_new, v_new
        present = (k, v) if use_cache else None      # return the updated cache (still G kv heads) for the next step to use
        T_kv = k.size(2)                              # total key/value length to attend to right now (including history)
        past_len = T_kv - T_q                         # history length; the absolute position of the new query starts at past_len

        # only "when it's actually used" do we replicate the G copies into H copies to align with query (the cache always holds only G copies, taking no extra GPU memory)
        k_rep = k.repeat_interleave(self.num_queries_per_kv, dim = 1) # (B, H, T_kv, head_size)
        v_rep = v.repeat_interleave(self.num_queries_per_kv, dim = 1)

        w = q @ k_rep.transpose(-2, -1) * (self.head_size ** -0.5)    # (B, H, T_q, T_kv)
        # causal mask: the i-th new query (absolute position past_len+i) can only see key positions ≤ past_len+i
        # take mask's sub-block [past_len:past_len+T_q, :T_kv]; in a full forward pass it is just the original [:T,:T], while in single-token decoding it is "can see all history + itself"
        w = w.masked_fill(self.mask[past_len:past_len + T_q, :T_kv] == 0, float("-Inf"))
        w = F.softmax(w, dim = -1)
        out = w @ v_rep # (B, H, T_q, head_size)

        out = out.permute(0, 2, 1, 3) # (B, T_q, H, head_size)
        out = out.reshape(batch_size, T_q, self.num_heads * self.head_size)
        out = self.fc(out) # (B, T_q, n_embed)

        # with cache off (training/evaluation) it returns the same as the original, only out; with cache on it additionally returns the updated cache present
        return (out, present) if use_cache else out

group_query_attn = GroupQueryAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,   # 8 query heads
    num_kv_heads = 2,        # use only 2 kv heads: 8 query heads split into 2 groups, each group of 4 shares 1 kv head (num_kv_heads=num_heads means MHA, =1 means MQA)
    block_size_max = block_size,
)

In [ ]:
print("shape of input of group-query attention module: {}".format(x.shape))
print("shape of output of group-query attention module: {}".format(group_query_attn(x).shape))

# query is projected to num_heads heads, while key/value are only projected to num_kv_heads heads
print("num_heads (query) = {}, num_kv_heads (key/value) = {}".format(group_query_attn.num_heads, group_query_attn.num_kv_heads))
print("KV cache shrinks relative to MHA to {}/{} = {:.2f} of the original".format(group_query_attn.num_kv_heads, group_query_attn.num_heads, group_query_attn.num_kv_heads / group_query_attn.num_heads))

In [ ]:
# GQA KV-cache incremental-inference demo: when generating token by token, cache only the K/V of G kv-heads, appending at each step,
# and verify the result is exactly identical to "recomputing the whole segment" (only this truly applies GQA's GPU-memory savings to inference)
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # assume these are the features of the T tokens arriving one by one

with torch.no_grad():
    # Approach A (reference): a single forward pass over the whole segment (this is exactly how training/evaluation runs)
    full = group_query_attn(seq)                               # (1, T, n_embed)

    # Approach B (KV cache incremental decode): each step feeds in only "the single newly-arrived token", caches past K/V, appending step by step
    past, steps = None, []
    for t in range(T):
        out_t, past = group_query_attn(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)                                    # each step outputs only the current token
    incr = torch.cat(steps, dim=1)                             # (1, T, n_embed)

print("KV-cache incremental == full recompute:", torch.allclose(full, incr, atol=1e-5),
      "| max error {:.1e}".format((full - incr).abs().max().item()))

# the K cached in past is G kv-heads (not H) -- this is exactly the GPU memory GQA saves
past_k, past_v = past
print("cached K shape (B, G, T, head_size):", tuple(past_k.shape),
      "→ stores only G={} kv-heads, not H={}".format(group_query_attn.num_kv_heads, group_query_attn.num_heads))
print("per-token KV cache size: GQA = 2*G*head_size = {} values; MHA is 2*H*head_size = {} values (about {:.0f}x smaller)".format(
    2 * group_query_attn.num_kv_heads * head_size, 2 * group_query_attn.num_heads * head_size,
    group_query_attn.num_heads / group_query_attn.num_kv_heads))

## 5.5. Multi-Head Latent Attention

**Multi-head Latent Attention (MLA)** is a KV-cache-saving scheme proposed by DeepSeek-V2 / V3 that is more aggressive than GQA. GQA's idea is to "reduce the number of KV heads" ($H \to G $, $H$: the number of heads, $G$: the group size); MLA takes a different angle: **it jointly compresses each token's K and V into a low-rank "latent vector" $c^{KV}$, and at inference time caches only this small vector**.

### 5.5.1. Core idea: low-rank compression + caching only the latent vector

For each token's input $x$:

- **Down-projection (compression)**: $c^{KV} = x W_{DKV}$, where the dimension $d_c$ is very small ($d_c \ll 2 H d_h$). **What gets cached at inference time is exactly this $c^{KV}$**.
- **Up-projection (reconstruction)**: when needed, each head's key and value are reconstructed from $c^{KV}$: $k_i = c^{KV} W_{UK}^{(i)}$ and $v_i = c^{KV} W_{UV}^{(i)}$.
- The query side also goes through a low-rank step ($x \to c^{Q} \to q$), but that is mainly to save **training** GPU memory; the credit for saving KV cache goes to $c^{KV}$.

Intuition: the information in K and V is highly redundant, so a single low-rank bottleneck $c^{KV}$ is enough to "hold" it; the reconstructed K and V are still full-rank and multi-head, with very little loss of expressiveness, yet what has to be cached drops sharply from $2 H d_h$ to $d_c$.

### 5.5.2. Two key details that make MLA actually usable

**(1) Decoupled RoPE**. RoPE multiplies the key by a position-dependent rotation $R_n$, so $k_n = R_n\,(W_{UK}\, c^{KV}_n)$ — this position-varying $R_n$ gets wedged between $c^{KV}$ and the query, making the "matrix absorption" below impossible (low-rank compression is incompatible with RoPE). DeepSeek's solution is to split both q and k into two parts:

- **Content part**: goes through the low-rank latent vector and **does not apply RoPE** (can be absorbed);
- **Positional part**: separately computes a small RoPE-carrying query / key, where **the RoPE key is shared by all heads** (similar to MQA) and cached separately.

The attention score = content-part dot product + positional-part dot product. RoPE lives only on that small decoupled dimension and never touches the compressible content path.

**(2) Matrix absorption** — this is the step that truly turns "caching only a small vector" into "saving both GPU memory and computation". At its core is an identity that **only changes the order of matrix multiplication**:

$$
q_i \cdot \big(W_{UK}^{(i)} c^{KV}\big) = \big(W_{UK}^{(i)\top} q_i\big) \cdot c^{KV}
$$

**Looking at a single forward pass, this really is just a reordering of the multiplications and saves no FLOPs**. The key saving lives in **the autoregressive decoding + KV cache loop**, and that loop has one important asymmetry: **each step has only 1 new query, yet the cache holds $T$ tokens**.

- **Without absorption (left-hand side, reconstruct K first)**: you must apply $W_{UK}$ to **every cached token's $c^{KV}$** to reconstruct K — with $T$ cached tokens that means $T$ up-projections; and since only $c^{KV}$ is cached and K is not, **every decode step has to redo these $T$ operations from scratch**.
- **With absorption (right-hand side, move $W_{UK}$ onto the query)**: $\tilde q_i = W_{UK}^{(i)\top} q_i$ acts only on **that 1 new query** (once per step, independent of $T$), and $\tilde q$ is then dot-multiplied directly with the cached $c^{KV}$.

**What you save is exactly that $\times T$**: the up-projection moves from "acting on $T$ cached keys, redone every step" to "acting on 1 new query, once per step". The output side works the same way:

$$
\sum_n a_n \big(W_{UV}^{(i)} c^{KV}_n\big) = W_{UV}^{(i)} \Big(\sum_n a_n\, c^{KV}_n\Big)
$$

The left side first reconstructs each of the $T$ $c^{KV}$ vectors into V and then weights them; the right side instead **first aggregates the $c^{KV}$ by weight into 1 context vector in the latent-vector space, and finally up-projects only this single one** — again moving $W_{UV}$ from "acting on $T$ cached tokens" to "acting on only 1 result".

Even more crucial is **GPU memory and bandwidth**: since K/V are never reconstructed at any point, the cache stores only that $d_c$-dimensional $c^{KV}$ (plus the shared RoPE key). And autoregressive decoding is inherently a **GPU-memory-bandwidth bottleneck** (every step must read the entire KV cache from GPU memory), so a cache an order of magnitude smaller is read an order of magnitude faster — and that is the main reason MLA accelerates inference in practice.

In one sentence: absorption is not "reordering for nothing" — it exploits the decoding-time asymmetry of "few queries, many cached key/value" to move the up-projection to **the smaller side**, eliminating the computation that would otherwise be **repeated every step on the larger side**, and shrinking the KV cache to its minimum along the way.

### 5.5.3. KV cache size comparison

| Scheme | What is cached per token | KV cache size |
|---|---|---|
| **MHA** | full K, V | $2 H d_h$ |
| **GQA** | K, V shared across $G$ groups | $2 G d_h$ |
| **MLA** | low-rank latent vector $c^{KV}$ (+ one shared RoPE key) | $d_c + d_r$ (minimal) |

> The `MultiHeadLatentAttn` below implements all three of these points (low-rank compression + decoupled RoPE + matrix absorption); `forward(x, past_kv, use_cache)` supports **KV-cache incremental decoding**, storing only $c^{KV}$ and that shared RoPE key in the cache. The demo verifies that "incremental decoding == full recomputation" and prints out that the cache really contains only these two small vectors.
>
> There are still teaching simplifications: RoPE uses the most naive split-half formulation; the real DeepSeek-V3 further merges $W_{UK}/W_{UV}$ offline into $W_Q/W_O$ for real and applies engineering optimizations such as quantizing the cache. Here it is enough to make the mechanism of "decoupling + absorption + caching only the latent vector" clear.

In [ ]:
# Full MLA implementation: low-rank compression (c_KV) + decoupled RoPE + matrix absorption + KV-cache incremental decoding
def apply_rope(t, start_pos = 0):
    # Rotate the last dim of t by "absolute positions start_pos..start_pos+T-1" (split-half convention; dim must be even)
    # start_pos lets new tokens use the correct positions during incremental decoding, consistent with the full-sequence forward pass
    T, d = t.shape[-2], t.shape[-1]
    half = d // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device = t.device).float() / half))
    ang = torch.outer(torch.arange(start_pos, start_pos + T, device = t.device).float(), inv)  # (T, half)
    cos, sin = ang.cos(), ang.sin()
    t1, t2 = t[..., :half], t[..., half:]
    return torch.cat([t1 * cos - t2 * sin, t1 * sin + t2 * cos], dim = -1)

class MultiHeadLatentAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,           # per-head dimension d_h of the content segment
        num_heads,           # number of heads H
        block_size_max,
        kv_latent_dim,       # KV latent-vector dim d_c: inference caches only this, the smaller the cheaper (MLA's core hyperparameter)
        q_latent_dim = None, # query goes low-rank too (mainly saves training GPU memory), defaults to the same as kv_latent_dim
        rope_head_dim = 16,  # dim d_r of the decoupled RoPE segment (very small, must be even)
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        self.kv_latent_dim = kv_latent_dim
        self.rope_head_dim = rope_head_dim
        q_latent_dim = q_latent_dim or kv_latent_dim

        # query: x -> low-rank c_Q -> per head [content segment q_c(head_size) + position segment q_r(rope_head_dim)]
        # Note: the q_down→q_up below is a "low-rank factorization" of the query projection. There is no nonlinearity between the two layers,
        #   so it is mathematically equivalent to a single Linear(n_embed → num_heads*(head_size+rope_head_dim)) and can be fused into one layer.
        #   Real MLA keeps it to save [training] GPU memory/parameters when q_latent_dim ≪ n_embed (unrelated to the KV cache——q is not cached);
        #   this tutorial version sets q_latent_dim=n_embed, so there is no compression effect (the factorization actually adds a few hundred parameters); it is kept only to mirror the structure of real MLA.
        self.q_down = nn.Linear(n_embed, q_latent_dim, bias = False)
        self.q_up   = nn.Linear(q_latent_dim, num_heads * (head_size + rope_head_dim), bias = False)
        # Core: K and V are jointly compressed into one low-rank latent vector c_KV(kv_latent_dim)——this is exactly what gets cached at inference time
        self.kv_down = nn.Linear(n_embed, kv_latent_dim, bias = False)
        self.k_up    = nn.Linear(kv_latent_dim, num_heads * head_size, bias = False)  # W_UK: reconstruct c_KV into each head's K
        self.v_up    = nn.Linear(kv_latent_dim, num_heads * head_size, bias = False)  # W_UV: reconstruct c_KV into each head's V
        # Decoupled RoPE key: project a small segment directly from x with RoPE, [shared across all heads] (similar to MQA), also cached
        self.k_rope  = nn.Linear(n_embed, rope_head_dim, bias = False)
        self.fc      = nn.Linear(num_heads * head_size, n_embed)                      # W_O
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def forward(self, x, past_kv = None, use_cache = False):
        # x: (B, T_q, n_embed)
        #   · training / full-sequence forward pass: past_kv=None, use_cache=False
        #   · inference incremental decoding: x feeds only the new token, past_kv passes in the previous step's cache (c_KV and the shared RoPE key), use_cache=True
        B, T_q, _ = x.size()
        H, dh, dc, dr = self.num_heads, self.head_size, self.kv_latent_dim, self.rope_head_dim
        past_len = 0 if past_kv is None else past_kv[0].size(1)   # number of cached tokens = starting absolute position of the new token

        # —— This step's new token: compress out the low-rank latent vector c_KV and a small shared decoupled RoPE key (both tiny; inference caches only these two) ——
        c_kv_new = self.kv_down(x)                        # (B, T_q, dc)
        k_r_new  = apply_rope(self.k_rope(x), past_len)   # (B, T_q, dr)  shared across heads
        if past_kv is not None:                           # append to the cache (history is not recomputed)
            c_kv = torch.cat([past_kv[0], c_kv_new], dim = 1)
            k_r  = torch.cat([past_kv[1], k_r_new],  dim = 1)
        else:
            c_kv, k_r = c_kv_new, k_r_new
        present = (c_kv, k_r) if use_cache else None      # the cache stores only c_KV and the shared RoPE key
        T_kv = c_kv.size(1)

        # —— query: after low-rank reconstruction, split into content segment / position segment; add RoPE to the position segment ——
        q = self.q_up(self.q_down(x)).view(B, T_q, H, dh + dr)
        q_c = q[..., :dh].transpose(1, 2)                       # (B,H,T_q,dh) content segment
        q_r = apply_rope(q[..., dh:].transpose(1, 2), past_len) # (B,H,T_q,dr) position segment (with RoPE)

        # —— Matrix absorption ①: q_c·(W_UK·c_KV) = (W_UK^T·q_c)·c_KV, fold W_UK into the query and score directly against the cached c_KV without reconstructing K ——
        W_UK = self.k_up.weight.view(H, dh, dc)
        q_tilde = torch.einsum("bhtd,hdc->bhtc", q_c, W_UK)          # q_tilde = W_UK^T q_c, (B,H,T_q,dc)
        score_content = torch.einsum("bhtc,bsc->bhts", q_tilde, c_kv)
        score_rope    = torch.einsum("bhtd,bsd->bhts", q_r, k_r)     # position segment (shared RoPE key)
        w = (score_content + score_rope) * ((dh + dr) ** -0.5)       # (B,H,T_q,T_kv)
        w = w.masked_fill(self.mask[past_len:past_len + T_q, :T_kv] == 0, float("-inf"))
        w = F.softmax(w, dim = -1)

        # —— Matrix absorption ②: Σ a·(W_UV·c_KV) = W_UV·(Σ a·c_KV), aggregate first in the latent-vector space, then up-project only once at the end, without reconstructing the full V sequence ——
        ctx  = torch.einsum("bhts,bsc->bhtc", w, c_kv)              # (B,H,T_q,dc)
        W_UV = self.v_up.weight.view(H, dh, dc)
        out  = torch.einsum("bhtc,hdc->bhtd", ctx, W_UV)            # out = W_UV · ctx, (B,H,T_q,dh)
        out  = self.fc(out.transpose(1, 2).reshape(B, T_q, H * dh))
        return (out, present) if use_cache else out

multi_head_latent_attn = MultiHeadLatentAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
    kv_latent_dim = 32,   # compress K and V to a 32-dim latent vector (far smaller than the 2*8*16=256 that MHA must cache)
    rope_head_dim = 16,
)

In [ ]:
# MLA KV-cache incremental inference: when generating token by token, cache only the "latent vector c_KV + shared RoPE key" (both very small), appending at each step,
# and verify it matches "full recomputation" element-by-element — while clearly revealing this MLA advantage: "throughout, only the small latent vector is cached, and K/V are never explicitly reconstructed"
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # features of the T tokens assumed to arrive step by step

with torch.no_grad():
    full = multi_head_latent_attn(seq)                        # single full-sequence forward pass (reference)
    past, steps = None, []
    for t in range(T):
        out_t, past = multi_head_latent_attn(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)
    incr = torch.cat(steps, dim=1)

print("MLA KV-cache incremental == full recomputation:", torch.allclose(full, incr, atol=1e-5),
      "| max error {:.1e}".format((full - incr).abs().max().item()))

c_kv_cache, k_rope_cache = past
print("Cache contents: latent vector c_KV {} + shared RoPE key {}".format(tuple(c_kv_cache.shape), tuple(k_rope_cache.shape)))
mla_per_tok = multi_head_latent_attn.kv_latent_dim + multi_head_latent_attn.rope_head_dim
mha_per_tok = 2 * num_heads * head_size
print("Per-token KV cache: MLA = d_c + d_r = {} dims  vs  MHA = 2*H*head_size = {} dims (about {:.1f}x smaller)".format(
    mla_per_tok, mha_per_tok, mha_per_tok / mla_per_tok))
print("Note: throughout the process only this small latent vector is cached/read; K and V are never explicitly reconstructed — attention is computed directly on c_KV via matrix absorption.")

## 5.6. Feed-Forward Layer

After the Attention module we add a Feed-Forward Layer. The Attention module lets features at different positions interact with one another, but this interaction is obtained as a weighted sum of the feature at each position with the features at other positions — a series of linear transformations — so its ability to capture nonlinear relationships is limited. We therefore introduce the Feed-Forward module, which uses one linear transformation to expand the dimension, passes the result through an activation function, and then uses another linear transformation to restore the original dimension. This two-stage linear-transformation design lets the Feed-Forward Layer extract deeper features, while the nonlinear activation function increases the model's expressive power. It is worth noting that although the FFN is structurally simple, it accounts for the bulk of a Transformer's parameters and computation (about 2/3) — it performs an independent nonlinear transformation at each position and is one of the main sources of the model's capacity and expressive power.

**Modern practice**: the activation used here is ReLU; modern LLMs more commonly use the smoother **GELU** (GPT-2 / BERT), or the gated **SwiGLU** (Llama 3, Qwen, DeepSeek, etc.; Google's Gemma uses the closely related GeGLU). SwiGLU splits the up-projection layer into two branches — one passed through SiLU as the "gate" and one as the "value" — multiplies them element-wise, and then projects back down: $\text{SwiGLU}(x) = \big(\text{SiLU}(xW_1) \odot xW_3\big)W_2$, which is more expressive; to keep the parameter count comparable despite the extra branch, its intermediate dimension is usually set to $\tfrac{8}{3}d$ rather than $4d$.

In [ ]:
class FeedForward(nn.Module):
    def __init__(
        self,
        n_embed,
        scaling_factor = 4,
    ):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.ReLU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
        )
        
    def forward(self, x):
        return self.seq(x)

feed_forward = FeedForward(n_embed = n_embed, scaling_factor = 4)

In [ ]:
print("shape of input: {}".format(x.shape))
print("shape of output of feed forward module: {}".format(feed_forward(multi_head_attn(x)).shape))

In [ ]:
# Modern practice: two common FFN alternatives (compared with the ReLU version above)
class FeedForwardGELU(nn.Module):
    # Just swap ReLU for the smoother GELU (GPT-2 / BERT)
    def __init__(self, n_embed, scaling_factor=4):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.GELU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
        )
    def forward(self, x):
        return self.seq(x)

class FeedForwardSwiGLU(nn.Module):
    # Gated version (SwiGLU, widely adopted by Llama 3 / Qwen / DeepSeek, etc.): one branch passes through SiLU as the "gate", the other serves as the "value", multiplied element-wise and then projected down
    def __init__(self, n_embed, hidden=None):
        super().__init__()
        hidden = hidden or int(8 / 3 * n_embed)           # one extra branch, so the hidden dim is set to ~8/3·d, keeping total params on par with a plain 4·d FFN
        self.w1 = nn.Linear(n_embed, hidden, bias=False)  # gate branch
        self.w3 = nn.Linear(n_embed, hidden, bias=False)  # value branch
        self.w2 = nn.Linear(hidden, n_embed, bias=False)  # project back down to n_embed
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))   # SiLU(xW1) ⊙ xW3 → W2

# All three produce identical output shapes, with comparable parameter counts too (SwiGLU uses 8/3·d to offset the cost of the "extra branch")
for name, m in [("ReLU   (4d)", FeedForward(n_embed)),
                ("GELU   (4d)", FeedForwardGELU(n_embed)),
                ("SwiGLU (~8/3 d)", FeedForwardSwiGLU(n_embed))]:
    print("{:16s} out={}  params={}".format(name, tuple(m(x).shape), sum(p.numel() for p in m.parameters())))

**More cutting-edge (MoE / Mixture-of-Experts)**: DeepSeek-V3, Qwen3-MoE, Mixtral, and others replace this dense FFN with a **Mixture-of-Experts (MoE)**—placing many FFN "experts" side by side and adding a "router" network that scores each token, so that **each token activates only its top-k experts** (e.g., DeepSeek-V3 selects 8 out of 256, plus several shared experts). Thus the model's **total parameters (capacity) can be made enormous, while the actual computation per token stays almost unchanged**—this is the mainstream approach for scaling up open-source models today. Below we directly implement **sparse dispatch**: group tokens by their selected expert, have each expert compute only the tokens assigned to it, then use `index_add_` to merge the outputs according to the routing weights. The expert-FFN computation per token is proportional to `top_k`; increasing the total number of experts still adds overhead to router scoring and dispatch.

In [ ]:
# Modern practice (MoE): sparse dispatch, each expert only computes the tokens routed to it
class MoEFeedForward(nn.Module):
    def __init__(self, n_embed, num_experts=4, top_k=2):
        super().__init__()
        self.experts = nn.ModuleList([FeedForward(n_embed) for _ in range(num_experts)])
        self.router = nn.Linear(n_embed, num_experts, bias=False)
        self.top_k = top_k
    def forward(self, x):
        B, T, d = x.shape
        xf = x.reshape(-1, d)                                   # (N, d), N=B*T, flatten all tokens
        topw, topi = self.router(xf).topk(self.top_k, dim=-1)   # (N, top_k)
        topw = F.softmax(topw, dim=-1)
        # Flatten into a "dispatch table": N*top_k rows of (token index, selected expert index, weight)
        tok = torch.arange(xf.size(0), device=x.device).repeat_interleave(self.top_k)  # (N*top_k,)
        exp = topi.reshape(-1)                                  # (N*top_k,)
        wgt = topw.reshape(-1, 1)                               # (N*top_k, 1)
        out = torch.zeros_like(xf)
        for e, expert in enumerate(self.experts):               # each expert only processes the batch of tokens "assigned to it"
            sel = exp == e
            if sel.any():
                t = tok[sel]                                    # indices of these tokens
                out.index_add_(0, t, expert(xf[t]) * wgt[sel])  # after computing, scatter-add back weighted by the gate weights
        return out.view(B, T, d)

moe = MoEFeedForward(n_embed, num_experts=4, top_k=2)
print("MoE output shape:", tuple(moe(x).shape))
dense = sum(p.numel() for p in FeedForward(n_embed).parameters())
moe_total = sum(p.numel() for p in moe.parameters())
print("Dense FFN params={} | 4-expert MoE total params={} (~{:.0f}x), but each token only passes through top_k={} experts".format(
    dense, moe_total, moe_total / dense, moe.top_k))

**How industrial-grade implementations do it**: the sparse implementation above still keeps a `for e in experts` Python loop, with one small matmul per expert. Production frameworks go further:

- **Grouped GEMM (grouped / segment GEMM)**: after sorting tokens by expert, use a single "segmented matrix multiply" to compute all experts at once, eliminating the Python loop (e.g., `grouped_gemm`, ScatterMoE).
- **Capacity + batched bmm (GShard style)**: give each expert a fixed capacity, pad tokens into `(num_experts, capacity, d)`, and compute them in parallel with a single `bmm`; tokens beyond the capacity are dropped (an approximation, traded for regular tensor shapes).
- **Block-sparse / fused kernels**: `Megablocks` uses block-sparse GEMM to achieve "no dropped tokens while staying efficient"; the **Triton `fused_moe`** in vLLM / SGLang fuses routing → grouped GEMM → weighted combination into just a handful of GPU kernels — this is exactly what is usually called **"fused MoE"**.
- **Expert parallelism (EP)**: when there are many experts, place different experts on different GPUs and use all-to-all to send tokens over and gather them back (standard practice for large-scale MoE such as DeepSeek-V3).

The sparse dispatch above removes redundancy at the **algorithmic level** (no longer computing the experts that were not selected); industrial implementations then layer **kernel fusion + parallelism** on top, which is what truly turns MoE's "large capacity, small compute" advantage into real speed.

## 5.7. Add & LayerNorm

When a deep neural network has too many layers, it suffers degradation caused by vanishing or exploding gradients; the Add (residual connection) lets information skip across one or more layers, thereby alleviating the vanishing- and exploding-gradient problems.
Before introducing LayerNorm, let's first answer: why do we need normalization? Because it works. Experiments find that normalization can smooth gradients, speed up training, and improve model accuracy, but there is no solid theoretical explanation. For instance, one explanation is that normalization suppresses the Internal Covariate Shift problem — that is, every layer of a deep network has to adapt to the shifting distributions of the layers before it, which makes learning in deep layers difficult. Normalization, in turn, keeps each layer's input distribution within a certain range, thereby lowering the difficulty of learning. The CNN era used BatchNorm. But the LLM era switched to LayerNorm: BatchNorm depends on the statistics of the entire batch and is very unstable when the batch is small or during autoregressive token-by-token inference (its training/inference behavior is also inconsistent); LayerNorm, by contrast, normalizes each token independently along the feature (dim) dimension, does not depend on the batch, and behaves consistently in training and inference. (Variable-length sequences / the need for padding is just one of the factors that make batch-dimension statistics more troublesome.) But whatever the norm, everyone has found it useful experimentally, yet cannot explain why it is useful.

A note on ordering: the original Transformer is **Post-LN** ($\text{LayerNorm}(x + \text{Sublayer}(x))$, add first, then normalize); whereas the Block used when we build the model below adopts **Pre-LN**, which is more common in modern LLMs ($x + \text{Sublayer}(\text{LayerNorm}(x))$, normalize first, then go through the sublayer, and finally add the residual). It trains more stably and usually needs no learning-rate warmup. In addition, after all the Blocks are stacked, one more final LayerNorm is appended, normalizing once again the features that are fed into the output layer.

**Modern practice**: Modern LLMs (Llama 3, Qwen, DeepSeek, Gemma, etc.) mostly use **RMSNorm** in place of LayerNorm — it drops the "mean subtraction" and the bias and scales only by the root mean square: $\text{RMSNorm}(x) = \dfrac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \odot g$ (where $g$ is a learnable per-dimension gain). It is more compute-efficient, and in practice performs on par with LayerNorm or even more stably.

To summarize: today, **RoPE + GQA + RMSNorm (pre-norm) + SwiGLU** has become a fairly uniform "standard foundation" for open-source large models (Llama 3, Qwen, Gemma, Mistral, etc.); while more cutting-edge open-source models (DeepSeek-V3, Qwen3-MoE, etc.) build further on top of this with techniques such as **MoE** (to expand capacity, see above) and **MLA** (to compress the KV cache, see above).

In [ ]:
class CheckLayerNorm(nn.Module):
    def __init__(
        self,
        n_embed
    ):
        super().__init__()
        self.ln = nn.LayerNorm(n_embed)
    
    def forward(
        self, x
    ):
        return self.ln(x)

check_layer_norm = CheckLayerNorm(n_embed = n_embed)

In [ ]:
# Observe the normalization effect of LayerNorm on the feature dimension
# x: (batch_size, block_size, n_embed)
ith_sample = 0
ith_token = 0
print("for {}th sample and {}th token in x, the var of feature dim is {} and the mean of feature dim is {}".format(ith_sample, ith_token, x[ith_sample, ith_token].var(), x[ith_sample, ith_token].mean()))
x_after_ln = check_layer_norm(x)
print("after layer normalization, the var of feature dim is {} and the mean of feature dim is {}".format(x_after_ln[ith_sample, ith_token].var(), x_after_ln[ith_sample, ith_token].mean()))

In [ ]:
# Modern practice: RMSNorm (Llama 3 / Qwen / DeepSeek / Gemma, etc.) vs LayerNorm — scales by root-mean-square only, does not subtract the mean, no bias
class RMSNorm(nn.Module):
    def __init__(self, n_embed, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(n_embed))    # learnable per-dimension gain (no bias, and does not subtract the mean)
    def forward(self, x):
        ms = x.pow(2).mean(dim=-1, keepdim=True)       # compute the "mean square" (mean of squares) along the feature dimension
        return x * torch.rsqrt(ms + self.eps) * self.g

rms_norm = RMSNorm(n_embed)
ln = nn.LayerNorm(n_embed)
x_ln, x_rms = ln(x), rms_norm(x)
i = (0, 0)   # look at the feature dimension of sample 0, token 0
print("Input           : mean={:.3f} variance={:.3f}".format(x[i].mean().item(), x[i].var(unbiased=False).item()))
print("After LayerNorm : mean={:.3f} variance={:.3f}  params={} (weight+bias, both centers and scales)".format(
    x_ln[i].mean().item(), x_ln[i].var(unbiased=False).item(), sum(p.numel() for p in ln.parameters())))
print("After RMSNorm   : mean={:.3f} RMS={:.3f}  params={} (gain only, does not subtract the mean, so the mean need not be 0)".format(
    x_rms[i].mean().item(), x_rms[i].pow(2).mean().sqrt().item(), sum(p.numel() for p in rms_norm.parameters())))

# 6. Building the Model

To actually train an LLM, beyond the model itself, there is a set of techniques that are "almost indispensable in engineering practice":

- **Dropout (regularization)**: a standard Transformer adds dropout on the attention weights, in the FFN, before the residual addition, and after the embedding, to prevent overfitting. At toy scale you can default to `p=0`, and when needed just insert `nn.Dropout(p)` into the `Attn` / `FeedForward` / `Block` defined below.
- **Weight tying**: have the input token embedding and the output projection share the same table — i.e., set `self.linear_final.weight = self.token_embed_tab.weight`. This saves parameters (substantial when the BPE vocabulary is large) and often improves quality as well; the intuition is that "table-lookup encoding" and "decoding by scoring" are inverse operations of each other.
- **Weight initialization**: this tutorial uses PyTorch's default initialization; GPT-2, by contrast, uses a normal $N(0,\,0.02)$ and additionally scales the residual-branch projections by $1/\sqrt{2\,n_\text{layers}}$, which materially helps the stability of training deep networks.
- **Learning-rate schedule**: below we use a fixed `lr=1e-4`; modern LLMs almost all use **warmup + cosine annealing** (first a linear warmup to prevent early divergence, then a cosine decay to aid convergence).
- **Gradient clipping**: the training loop below already has a built-in `clip_grad_norm_` line; this is the most direct means of preventing exploding gradients, complementing the residual connections / normalization discussed above.
- **Special tokens (BOS / EOS / PAD)**: real LLMs use special tokens to mark the **start** of a sequence (BOS), its **end** (EOS, echoing the earlier discussion — generation stops once EOS is produced), and the **padding** used for in-batch alignment (PAD, ignored when computing the loss via `ignore_index`). This tutorial is fixed-length and character-level, generating from a fixed starting point to a fixed length, so these are omitted; switching to general-purpose / 

With the techniques above, let's build the model:

In [ ]:
class Attn(nn.Module):
    def __init__(
        self,
        n_embed, head_size, num_heads, block_size_max, dropout = 0.0
    ):
        super().__init__()
        self.head_size = head_size
        self.num_heads = num_heads
        self.n_embed = n_embed

        self.key = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.fc = nn.Linear(head_size * num_heads, n_embed)
        self.attn_dropout = nn.Dropout(dropout)   # applied to the attention weights (after softmax)
        self.resid_dropout = nn.Dropout(dropout)  # applied after the output projection (residual branch)
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def split_heads(self, h):
        batch_size, seq_length, total_embed_size = h.size()
        assert total_embed_size == self.head_size * self.num_heads
        h = h.view(batch_size, seq_length, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h

    def forward(self, x, k = None, v = None, mask = None, past_kv = None, use_cache = False):
        # Only project the current input; the historical K/V are already (B, H, T_past, head_size).
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        if past_kv is not None and not use_cache:
            raise ValueError("use_cache must be enabled when past_kv is passed")
        if (k is None) != (v is None):
            raise ValueError("k and v must be provided together")
        if k is None:
            k, v = self.key(x), self.value(x)
        elif k.shape != v.shape or k.ndim != 3 or k.shape[0] != batch_size or k.shape[2] != self.head_size * self.num_heads:
            raise ValueError("k/v must be identically shaped (B, T, H * head_size) projections")
        if use_cache and k.shape[1] != seq_length:
            raise ValueError("KV cache is used for self-attention; this step's Q/K/V must have the same number of tokens")
        k, v = self.split_heads(k), self.split_heads(v)
        q = self.split_heads(self.query(x))

        past_len = 0
        if past_kv is not None:
            if not isinstance(past_kv, (tuple, list)) or len(past_kv) != 2:
                raise ValueError("each layer's past_kv must be (past_k, past_v)")
            past_k, past_v = past_kv
            if not isinstance(past_k, torch.Tensor) or not isinstance(past_v, torch.Tensor):
                raise ValueError("past_k and past_v must be Tensors")
            if past_k.ndim != 4 or past_k.shape != past_v.shape or past_k.shape[:2] != (batch_size, self.num_heads) or past_k.shape[-1] != self.head_size:
                raise ValueError("KV cache shape must be (B, H, T_past, head_size)")
            if past_k.device != k.device or past_v.device != v.device or past_k.dtype != k.dtype or past_v.dtype != v.dtype:
                raise ValueError("the KV cache must match the current K/V in device and dtype")
            past_len = past_k.shape[2]
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        key_len = k.shape[2]
        if past_len + seq_length > self.mask.shape[0] or key_len > self.mask.shape[1]:
            raise ValueError("attention length exceeds block_size_max")
        w = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        if mask is not None:
            if mask.shape != (seq_length, key_len):
                raise ValueError("mask shape must be (T_new, T_past + T_new)")
        else:
            # Queries correspond to positions past_len ... past_len+T_new-1; appending multiple tokens at once is supported.
            mask = self.mask[past_len:past_len + seq_length, :key_len]
        w = w.masked_fill(mask == 0, float("-inf"))
        w = self.attn_dropout(F.softmax(w, dim=-1))
        out = w @ v
        out = out.permute(0, 2, 1, 3).reshape(batch_size, seq_length, self.head_size * self.num_heads)
        out = self.resid_dropout(self.fc(out))
        return (out, (k, v)) if use_cache else out


class FeedForward(nn.Module):
    def __init__(self, n_embed, scaling_factor = 4, dropout = 0.0):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.ReLU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
            nn.Dropout(dropout),   # identity when p=0 (the default)
        )

    def forward(self, x):
        return self.seq(x)


class Block(nn.Module):
    def __init__(
        self,
        n_embed, head_size, num_heads, block_size_max, dropout = 0.0
    ):
        super().__init__()
        self.multiheadattn = Attn(n_embed, head_size, num_heads, block_size_max, dropout = dropout)
        self.ffwd = FeedForward(n_embed, dropout = dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x, past_kv = None, use_cache = False):
        # Pre-Norm: what gets cached is the K/V projected from this layer's ln1(x).
        attn_out = self.multiheadattn(self.ln1(x), past_kv = past_kv, use_cache = use_cache)
        if use_cache:
            attn_out, present_kv = attn_out
        x = x + attn_out
        x = x + self.ffwd(self.ln2(x))
        return (x, present_kv) if use_cache else x


class Model(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        n_layers,
        head_size,
        num_heads,
        block_size,
        block_size_max,
        dropout = 0.0
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_tab = nn.Embedding(block_size_max, n_embed)
        self.drop = nn.Dropout(dropout)   # dropout after the embedding
        self.blocks = nn.Sequential(*[Block(n_embed = n_embed, head_size = head_size, num_heads = num_heads, block_size_max = block_size_max, dropout = dropout) for _ in range(n_layers)])
        self.layernorm_final = nn.LayerNorm(n_embed)
        self.linear_final = nn.Linear(n_embed, vocab_size)
        self.block_size = block_size

        # Weight tying: the input token embedding and the output projection share the same table
        # (both have shape (vocab_size, n_embed); intuitively, "encoding by table lookup" and "decoding by scoring" are inverse operations of each other)
        self.linear_final.weight = self.token_embed_tab.weight

        # Weight initialization: all Linear / Embedding use N(0, 0.02)
        self.apply(self._init_weights)
        # GPT-2 trick: additionally scale the residual branch's output projection by 1/sqrt(2*n_layers)
        for pn, p in self.named_parameters():
            if pn.endswith("fc.weight") or pn.endswith("seq.2.weight"):
                torch.nn.init.normal_(p, mean = 0.0, std = 0.02 * (2 * n_layers) ** -0.5)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

    def _cache_length(self, past_kv, batch_size):
        if past_kv is None:
            return 0
        if not isinstance(past_kv, (tuple, list)) or len(past_kv) != len(self.blocks):
            raise ValueError("past_kv must provide one K/V pair for each Transformer layer")
        lengths = []
        for block, layer_kv in zip(self.blocks, past_kv):
            if not isinstance(layer_kv, (tuple, list)) or len(layer_kv) != 2:
                raise ValueError("each layer's cache must be (K, V)")
            k, v = layer_kv
            attn = block.multiheadattn
            if not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor):
                raise ValueError("the cached K/V must be Tensors")
            if k.ndim != 4 or k.shape != v.shape or k.shape[:2] != (batch_size, attn.num_heads) or k.shape[-1] != attn.head_size:
                raise ValueError("each layer's KV cache shape must be (B, H, T_past, head_size)")
            lengths.append(k.shape[2])
        if lengths and any(length != lengths[0] for length in lengths):
            raise ValueError("all layers' KV cache lengths must be identical")
        return lengths[0] if lengths else 0

    def forward(self, x, target = None, past_kv = None, use_cache = False):
        if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
            raise ValueError("x must be a non-empty (B, T) token sequence")
        if target is not None and use_cache:
            raise ValueError("do not enable KV cache during training; use model(x, target)")
        if past_kv is not None and not use_cache:
            raise ValueError("use_cache must be enabled when past_kv is passed")
        B, T = x.shape
        past_len = self._cache_length(past_kv, B)
        if past_len + T > self.pos_embed_tab.num_embeddings:
            raise ValueError("total length exceeds the block_size_max of the learned positional encoding")
        tok_embed = self.token_embed_tab(x)
        # Positions for incremental input continue numbering from the historical cache length.
        pos_embed = self.pos_embed_tab(torch.arange(past_len, past_len + T, device = x.device))
        x = self.drop(tok_embed + pos_embed)
        if use_cache:
            presents = []
            for layer_idx, block in enumerate(self.blocks):
                layer_past = None if past_kv is None else past_kv[layer_idx]
                x, layer_present = block(x, past_kv = layer_past, use_cache = True)
                presents.append(layer_present)
        else:
            x = self.blocks(x)
        logits = self.linear_final(self.layernorm_final(x))

        if target is None:
            loss = None
        else:
            assert target.shape == logits.shape[:-1]
            loss = F.cross_entropy(logits.view(B*T, -1), target.view(B*T))
        return (logits, loss, tuple(presents)) if use_cache else (logits, loss)

    def decode_step(self, context, past_kv = None, use_cache = True, block_size = None):
        """KV cache by default; returns (B, V) logits and the cache.

        past_kv must come from the previous step of the same request, where context has appended exactly one token at the end;
        pass None for a new request or when modifying existing context. The first step is a prefill, after which only the newly added token is fed in.
        """
        if context.ndim != 2 or context.shape[0] == 0 or context.shape[1] == 0:
            raise ValueError("context must be a non-empty (B, T) token sequence")
        window = self.block_size if block_size is None else block_size
        if not isinstance(window, int) or not 0 < window <= self.pos_embed_tab.num_embeddings:
            raise ValueError("block_size must be in [1, block_size_max]")
        if not use_cache:
            logits, _ = self(context[:, -window:])
            return logits[:, -1, :], None
        past_len = self._cache_length(past_kv, context.shape[0])
        if past_len > window:
            raise ValueError("the cache length exceeds the current inference window; restart from past_kv=None")
        if past_kv is not None:
            if past_len < window and context.shape[1] != past_len + 1:
                raise ValueError("context must append exactly one token to the end of the context corresponding to the cache; pass past_kv=None for a new request")
            if past_len == window and context.shape[1] <= window:
                raise ValueError("the cache window is full; context must contain the new token plus the complete sliding window")
        if past_kv is not None and past_len + 1 <= window:
            logits, _, presents = self(context[:, -1:], past_kv = past_kv, use_cache = True)
        else:
            # Rebuild on the first step or when the window is full. Positions reset from 0, and the deep-layer K/V also depend on the discarded context,
            # so we cannot merely trim the old cache; only a rebuild stays strictly consistent with the original sliding-window inference.
            logits, _, presents = self(context[:, -window:], use_cache = True)
        return logits[:, -1, :], presents

    def generate(self, x, max_new_tokens = 100, use_cache = True, eos_token_id = None):
        """Uses KV cache by default; use_cache=False serves as a full-window-recompute reference."""
        if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
            raise ValueError("prompt must be a non-empty (B, T) token sequence")
        if not isinstance(max_new_tokens, int) or max_new_tokens < 0:
            raise ValueError("max_new_tokens must be a non-negative integer")
        if eos_token_id is not None and (not isinstance(eos_token_id, int) or not 0 <= eos_token_id < self.token_embed_tab.num_embeddings):
            raise ValueError("eos_token_id must be a token id within the vocabulary")
        was_training = self.training
        self.eval()
        try:
            with torch.inference_mode():
                past_kv = None
                finished = torch.zeros(x.shape[0], dtype=torch.bool, device=x.device)
                for _ in range(max_new_tokens):
                    logits, past_kv = self.decode_step(x, past_kv = past_kv, use_cache = use_cache)
                    probs = F.softmax(logits, dim=-1)
                    idx_next = torch.multinomial(probs, num_samples=1)
                    if eos_token_id is not None:
                        # Keep the batch rectangular: finished samples keep getting filled with EOS, and we exit once all are finished.
                        idx_next = idx_next.masked_fill(finished[:, None], eos_token_id)
                        finished |= idx_next[:, 0] == eos_token_id
                    x = torch.cat((x, idx_next), dim=1)
                    if eos_token_id is not None and finished.all():
                        break
        finally:
            self.train(was_training)
        return x

Beyond the model itself, the engineering tricks, and the input/output data, we still need a **loss function** to measure the difference between the model's current output and the target result. We use **cross-entropy** as the loss function (already implemented in the code above as `loss = F.cross_entropy(logits.view(B*T, -1), target.view(B*T))`).

## 6.1. Why use cross-entropy as the loss function?

At every position, the model outputs "the probability distribution of the next token over the entire vocabulary," $p_\theta(\cdot \mid \text{context}) = \mathrm{softmax}(\text{logits})$. The training objective is simply this: make the model assign as high a probability as possible to **the next token that actually appears**. Cross-entropy is precisely this statement written as an optimizable loss.

**It is essentially maximum likelihood estimation (MLE).** Denote the true next token at a given position as $y$; the cross-entropy loss at that position is

$$
\ell = -\log p_\theta(y \mid \text{context})
$$

Averaged over all $B\times T$ positions:

$$
\mathcal{L} = -\frac{1}{BT}\sum_{\text{all positions}} \log p_\theta(y \mid \text{context})
$$

Minimizing $\mathcal{L}$ $\Longleftrightarrow$ maximizing the log-likelihood the model assigns to the entire training data $\Longleftrightarrow$ **maximum likelihood estimation**. This is exactly the training-time version of the earlier $p(t_k \mid t_1,\dots,t_{k-1}, \theta)$: pushing as much probability mass as possible onto the token that actually appears.

**An information-theoretic view: here cross-entropy is simply the KL divergence.** Let the true distribution $q$ be one-hot (equal to 1 only at $y$) and the model distribution be $p$; then

$$
H(q, p) = -\sum_i q_i \log p_i = -\log p_y
$$

There is also the identity $H(q,p) = H(q) + D_{\mathrm{KL}}(q \parallel p)$, and the entropy of a one-hot distribution is $H(q)=0$, so in language modeling, where "the target is one-hot," **cross-entropy is exactly equal to the KL divergence** $D_{\mathrm{KL}}(q \parallel p)$ — minimizing it pulls the model distribution $p$ closer to the data distribution $q$. And $-\log p_y$ itself is "the **surprisal / information content** of the true token in the model's eyes"; training is just minimizing the average surprisal.

**Intuition:**
- Correct and confident ($p_y \to 1$) → loss $\to 0$; but **wrong yet confident ($p_y \to 0$) → loss $\to +\infty$** — it heavily penalizes being "confidently wrong," forcing the model to report its probabilities honestly.
- It watches only "how much probability the correct token receives" and ignores how the remaining tokens carve up the rest — a perfect fit for the next-token task.

**Why not MSE or accuracy?**
- Accuracy is not differentiable, so there is no way to backpropagate.
- Using MSE for classification, the gradient tends toward 0 in softmax's saturation region, so learning is very slow; whereas **the gradient of softmax + cross-entropy is especially clean**: $\partial \mathcal{L} / \partial\, \text{logit}_i = p_i - q_i$ (predicted probability − one-hot target), which is numerically stable and easy to optimize.

**perplexity $= e^{\mathcal{L}}$** is then the "plain-language version" of cross-entropy: ≈ how many candidates the model is, on average, torn between at each step (precisely the value printed by `val()` above). Below we verify each of these with a few lines of code:

In [ ]:
# Why cross-entropy: a few facts you can verify in one line each
torch.manual_seed(0)
V = 10                       # assume a vocabulary size of 10
logits = torch.randn(1, V, requires_grad=True)
target = torch.tensor([3])   # the true next token is index 3

# (1) cross-entropy = -log( probability the model assigns to the "correct token" ) — i.e. maximum likelihood
ce = F.cross_entropy(logits, target)
p  = F.softmax(logits, dim=-1)
print("F.cross_entropy    =", ce.item())
print("-log p[correct token] =", (-torch.log(p[0, target])).item(), " (the two are identical)")

# (2) the gradient of softmax + cross-entropy is especially clean: dL/d(logits) = p - onehot(target)
ce.backward()
onehot = F.one_hot(target, V).float()
print("gradient == p - onehot ?", torch.allclose(logits.grad, p - onehot, atol=1e-6))

# (3) here cross-entropy == KL(true one-hot || model p) (because the entropy of a one-hot is 0)
kl = (onehot * (torch.log(onehot + 1e-12) - torch.log(p))).sum()
print("cross-entropy == KL(true||model) ?", torch.allclose(ce, kl, atol=1e-6))

# (4) loss depends only on "the probability assigned to the correct token": more confident and correct→0; more confident and wrong→∞ (heavily penalizes confident mistakes)
for pc in [0.99, 0.5, 0.1, 0.01]:
    print("  p(correct)={:.2f} -> loss = -log p = {:.3f}".format(pc, -torch.log(torch.tensor(pc)).item()))

# (5) perplexity = exp(cross-entropy): ≈ on average how many candidates you're torn between
print("if mean cross-entropy = 2.0 -> perplexity = exp(2.0) =", round(torch.exp(torch.tensor(2.0)).item(), 2))

## 7. Training

The `Model` above actually implements two "modes of operation" at once, and it's worth pausing to compare them—this is the point beginners most often get confused about:

- **Training (`forward`, parallel, teacher forcing)**: a sequence of length $T$ is fed in, and in **a single forward pass** the model performs next-token prediction for all $T$ positions simultaneously, with the causal mask ensuring that position $i$ can only see itself and the real tokens before it. The condition at each position is the **true prefix** (rather than one the model guessed itself); this is called **teacher forcing**. The cross-entropy is averaged over all $B\times T$ positions, so a single sequence produces $T$ supervision signals, which makes training highly efficient.
- **Inference (`generate`, serial, autoregressive)**: there is no "ground truth" to feed in, so the only option is to **feed back** the token the model itself sampled onto the end of the input and then predict the next one, generating them one at a time (autoregressive). By default this model first processes the prompt in parallel (prefill), then generates incrementally, token by token, using the KV cache (decode); the new tokens in an ordinary autoregressive loop are still produced one after another. Earlier sampling results influence the subsequent outputs; later on we will also introduce speculative decoding, which can reduce the number of calls to the target model.

In one sentence: **at training time the model learns all positions in parallel while "looking at the answers," whereas at inference time it writes them out one at a time, serially and "with the answers covered up"**.

Now we formally begin training

init model

In [ ]:
model = Model(
    vocab_size,
    n_embed,
    n_layers,
    head_size,
    num_heads,
    block_size,
    block_size_max = block_size
)
model = model.to(device)

init optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

**Example: learning-rate warmup + cosine annealing**

The training above used a fixed `lr=1e-4`. Below is a drop-in schedule: first use `warmup_steps` steps to linearly ramp the lr from near 0 up to its peak (to prevent early divergence), then decay it along a cosine curve down to a minimum value. Let's plot the curve to see its shape:

In [ ]:
import math

def get_lr(step, max_lr=1e-4, min_lr=1e-5, warmup_steps=100, max_steps=2000):
    if step < warmup_steps:                                      # 1) linear warmup: ~0 -> max_lr
        return max_lr * (step + 1) / warmup_steps
    if step > max_steps:                                         # 2) hold the minimum lr afterwards
        return min_lr
    ratio = (step - warmup_steps) / (max_steps - warmup_steps)   # 3) cosine annealing: max_lr -> min_lr
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (max_lr - min_lr)

plt.figure()
plt.plot([get_lr(s) for s in range(2200)])
plt.xlabel("step"); plt.ylabel("lr"); plt.title("warmup + cosine schedule"); plt.show()

# How to hook this into the training loop: before each step, overwrite the optimizer's learning rate with the computed lr
#   lr = get_lr(global_step)
#   for g in optimizer.param_groups:
#       g["lr"] = lr

validation

In [ ]:
@torch.no_grad()
def val():
    losses = []
    model.eval()
    for batch in tqdm(test_datasetloader):
        logits, loss = model(
            batch["input"].to(device),
            batch["target"].to(device),
        )
        losses.append(loss.item())
    model.train()
    avg_loss = sum(losses) / len(losses)
    print(f"val perplexity: {torch.exp(torch.tensor(avg_loss)).item():.2f}")  # perplexity = exp(cross-entropy), the standard metric for LMs: ≈ "on average, how many candidates the model wavers between at each step"
    return avg_loss

def mov_avg(value, last_value, alpha = 0.99):
    return last_value * alpha + value * (1 - alpha)

In [ ]:
mov_avg_loss = None
for ith_epoch in range(num_epoch):
    for i, batch in enumerate(train_datasetloader):
        logits, loss = model(
            batch["input"].to(device),
            batch["target"].to(device),
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping: the standard safety net against exploding gradients
        optimizer.step()
        if mov_avg_loss is None:
            mov_avg_loss = mov_avg(loss.item(), loss.item())
        else:
            mov_avg_loss = mov_avg(loss.item(), mov_avg_loss)
        if (i+1) % print_interval == 0:
            print(f"epoch {ith_epoch+1} | step {i+1} | cur train loss: {loss.item()} | mov avg train loss: {mov_avg_loss}")

    print("start eval")
    val_loss = val()
    print(f"val loss: {val_loss}")

In [ ]:
# use_cache=True by default: incremental decoding within the window; rebuild once the window is exceeded (see below)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("[model output]:\n\n{}".format("".join([decode(i) for i in model.generate(context, max_new_tokens = 1000).detach().cpu().numpy()])))

In [ ]:
# Comparison version without KV cache; for default generation, call model.generate
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# max_new_tokens = 5000
# for _ in range(max_new_tokens):
#     with torch.no_grad():
#         logits, _ = model(context[:, -block_size:])
#     probs = F.softmax(logits[:, -1, :], dim=-1)
#     idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
#     context = torch.cat((context, idx_next), dim=1) # (B, T+1)

In [ ]:
# "".join([decode(i) for i in context.detach().cpu().numpy()])

# 8. Inference Techniques

## 8.1. Decoding Strategies

At each step, the model outputs a probability distribution over the entire vocabulary for the next token. So — how should we pick the `next token` from this distribution? Do we only ever pick the one with the highest probability? Below are several commonly used decoding strategies.

- **Greedy search**: at each step, take the token with the highest probability (`argmax`). It is highly deterministic and easy to reproduce, but it is also the most prone to falling into dull, repetitive loops (such as "the the the …"), because it never takes any risks.

- **Temperature**: before sampling, divide the logits by a temperature $T$ and then apply softmax. The larger $T$ is, the "flatter" the distribution, the more diverse the sampling, and the more likely surprising outputs become; the smaller $T$ is, the "sharper" the distribution and the closer to greedy ($T\to 0$ is greedy, $T=1$ is the original distribution).

- **Top-k sampling**: sample randomly, in proportion to their probabilities, only among the $k$ highest-probability candidates, zeroing out the rest. This preserves randomness while avoiding sampling absurd tokens from the long tail.

- **Nucleus / Top-p sampling**: instead of fixing the number of candidates, take "the smallest set of tokens whose cumulative probability just reaches $p$" — the size of this set adapts to the distribution (fewer candidates when the distribution is sharp, more when it is flat) — then sample within it in proportion to probability.

- **Beam search**: maintain $b$ candidate sequences (the "beams") at the same time, expanding each step by cumulative log-probability and keeping only the best $b$. It is commonly used for tasks that have a "relatively optimal solution," such as machine translation / summarization; for open-ended creative generation, on the other hand, it tends to be dull and repetitive, so modern LLMs rarely use it, and we will not expand on it here.

For a more systematic understanding, see [How to generate text: using different decoding methods for language generation with Transformers](https://huggingface.co/blog/how-to-generate)

## 8.2. The Generation Loop That Uses KV Cache by Default

**The `GreedySearch`, `TopkSampling`, `NucleusSampling`, and EOS examples in this section, as well as the earlier `Model.generate`, all default to `use_cache=True`.** The sampling strategy and the cache are independent of each other: the former decides how to pick tokens, while the latter avoids redundant computation. Explicitly passing `use_cache=False` lets you compare against recomputing over the whole window.

The first step, **prefill**, processes the prompt, builds the K/V cache for **each Transformer layer**, and takes the logits at the last position to select the first new token; afterward, **decode** feeds only the single new token generated in the previous step into the model. `Model.forward(..., past_kv=..., use_cache=True)` returns `(logits, loss, past_kv)`, and each layer's cache has shape `(B, num_heads, T_cached, head_size)`. Incremental positions are numbered continuing from the cache length, and the causal mask is sliced according to this offset; training still uses `model(x, target)` and does not cache by default.

The generation loop below also sets `model.eval()` and enters `torch.inference_mode()`: the former disables dropout, the latter disables gradient recording, and afterward the model is restored to the mode it was in before the call. Each request maintains its cache independently; the cache must correspond to the same unmodified prefix — you cannot continue generation using the KV from a different prompt.

**Context limitation of this tutorial**: the model uses learned absolute positional encoding, with a default `block_size=16`. The first step only processes the most recent `block_size` prompt tokens; within the window, KV is reused incrementally. Once the window is full, continuing to generate discards the earliest token and renumbers the remaining positions from 0; the positions and deep representations of the existing K/V no longer match, so `decode_step` **rebuilds the cache for the entire window**, keeping the same logits as the original sliding-window recomputation. Beyond the window you no longer get the per-token incremental speedup, and merely trimming the old KV cannot preserve identical results. Real LLMs typically keep reusing the cache within a much larger supported window; sliding-window models and long-term caching require corresponding designs for positions and attention.

In [ ]:
# The three strategies share one KV-cache generation loop; the sampling function only decides "how to pick the next token"
class _TextDecoder:
    def __init__(self, model, encode, decode, block_size, device):
        self.model = model
        self.encode = encode
        self.decode = decode
        self.block_size = block_size
        self.device = device

    def _generate(self, input_text_list, sample, max_new_tokens, use_cache, eos_token_id, desc):
        if isinstance(input_text_list, str):
            input_text_list = [input_text_list]
        if not isinstance(input_text_list, list) or not all(isinstance(t, str) for t in input_text_list):
            raise ValueError("input must be a string or a list of strings")
        if max_new_tokens < 0:
            raise ValueError("max_new_tokens must be non-negative")
        output_text_list = []
        was_training = self.model.training
        self.model.eval()  # turn off dropout; inference_mode separately disables gradient tracking
        try:
            with torch.inference_mode():
                # process one sequence at a time; this is a teaching loop, with multi-request parallel scheduling covered at the end of this section
                for i, text in enumerate(input_text_list):
                    context = torch.tensor(self.encode(text), dtype=torch.long, device=self.device).unsqueeze(0)
                    if context.shape[1] == 0:
                        raise ValueError("prompt needs at least one token; for empty input, explicitly add a BOS")
                    past_kv = None  # each prompt has its own cache; not shared across different requests
                    for _ in tqdm(range(max_new_tokens), desc=f"[{i+1}|{len(input_text_list)}] {desc}"):
                        next_token_logits, past_kv = self.model.decode_step(
                            context, past_kv=past_kv, use_cache=use_cache, block_size=self.block_size)
                        idx_next = sample(next_token_logits)  # (1, 1)
                        context = torch.cat((context, idx_next), dim=1)
                        if eos_token_id is not None and idx_next.item() == eos_token_id:
                            break
                    output_text_list.append(self.decode(context[0].tolist()))
        finally:
            self.model.train(was_training)
        return output_text_list


# Greedy: directly take the token with the largest logits; use_cache=True by default
class GreedySearch(_TextDecoder):
    def __call__(self, input_text_list, max_new_tokens=100, use_cache=True, eos_token_id=None):
        return self._generate(
            input_text_list, lambda logits: logits.argmax(dim=-1, keepdim=True),
            max_new_tokens, use_cache, eos_token_id, "decoding by greedy search")


greedy_search = GreedySearch(model, encode, decode, block_size, device)

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, greedy_search(input_text, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(greedy_search(input_text_list, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

In [ ]:
# Top-k sampling: use_cache=True by default, temperature must be > 0 (for greedy, use GreedySearch)
class TopkSampling(_TextDecoder):
    def __call__(self, input_text_list, top_k=5, temperature=1, max_new_tokens=100,
                 use_cache=True, eos_token_id=None):
        if top_k < 1 or temperature <= 0:
            raise ValueError("top_k must be >= 1, temperature must be > 0")

        def sample(logits):
            if top_k > logits.shape[-1]:
                raise ValueError("top_k cannot exceed the vocabulary size")
            # Set logits outside the top-k largest to -inf, then sample by temperature among the retained candidates
            kth_value = logits.topk(top_k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < kth_value, float('-inf'))
            probs = F.softmax(logits / temperature, dim=-1)
            return torch.multinomial(probs, 1)

        return self._generate(input_text_list, sample, max_new_tokens, use_cache,
                              eos_token_id, "decoding by top-k sampling")


topk_sampling = TopkSampling(model, encode, decode, block_size, device)

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, topk_sampling(input_text, top_k = 10, temperature = 1, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(topk_sampling(input_text_list, top_k = 10, temperature = 1, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

In [ ]:
# Nucleus / Top-p sampling: use_cache=True by default
class NucleusSampling(_TextDecoder):
    def __call__(self, input_text_list, p=0.9, temperature=1, max_new_tokens=100,
                 use_cache=True, eos_token_id=None):
        if not 0 < p <= 1 or temperature <= 0:
            raise ValueError("p must be in (0, 1], temperature must be > 0")

        def sample(logits):
            sorted_logits, sorted_idxs = logits.sort(dim=-1, descending=True)
            sorted_probs = F.softmax(sorted_logits / temperature, dim=-1)
            cumsum = sorted_probs.cumsum(dim=-1)
            sorted_remove = cumsum > p
            # Shift right by one to keep the "boundary token that just crosses p", and always keep at least the first candidate
            sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
            sorted_remove[:, 0] = False
            remove = torch.zeros_like(sorted_remove).scatter(1, sorted_idxs, sorted_remove)
            logits = logits.masked_fill(remove, float('-inf'))
            return torch.multinomial(F.softmax(logits / temperature, dim=-1), 1)

        return self._generate(input_text_list, sample, max_new_tokens, use_cache,
                              eos_token_id, "decoding by nucleus sampling")


nucleus_sampling = NucleusSampling(model, encode, decode, block_size, device)

**Example: stop at EOS (default KV cache)**

This tutorial is character-level by default and has no real end-of-sequence token; `Model.generate` and all three samplers support an optional `eos_token_id`, running for the full `max_new_tokens` when it is not passed. A real LLM should use the EOS or end-of-turn token specified by the tokenizer / model configuration. Below we reuse `Model.generate`'s cache loop, and at the character level we simply take the id of the **newline character** to demonstrate the stopping logic; it is not a real EOS. The samplers can also stop early using the same parameter.

In [ ]:
# Reuse the default KV-cache generation loop, avoiding a separate full-window recompute path for the EOS example
# idx is (1, T); Model.generate also supports an equal-length batch, padding finished samples with EOS
def generate_until_eos(model, idx, eos_token_id, max_new_tokens=200, use_cache=True):
    return model.generate(idx, max_new_tokens=max_new_tokens,
                          use_cache=use_cache, eos_token_id=eos_token_id)

# Both the character and BPE examples use the newline token; for a real model, use the EOS id from its config
newline_tokens = encode("\n")
if len(newline_tokens) != 1:
    raise ValueError("This example requires the newline to map to a single token; multi-token stop strings need separate detection")
eos_id = newline_tokens[0]
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = generate_until_eos(model, start, eos_id, max_new_tokens=200)
print("Default KV cache; generated until EOS (or the length limit), length =", out.shape[1])
print(repr(decode(out[0].tolist())))

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, nucleus_sampling(input_text, p = 0.3, temperature = 1, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(nucleus_sampling(input_text_list, p = 0.3, temperature = 1, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

## 8.3. Beyond Decoding: What Else Does Real LLM Inference Need?

This section has already covered the common **decoding strategies**: greedy, temperature, top-k, top-p, along with beam search and EOS. They decide "how the next token is chosen"; to cover mainstream **LLM inference techniques**, we also need to consider execution efficiency, request scheduling, and generation control. The KV cache, GQA, and MLA covered earlier are precisely the foundations along this line.

**First, distinguish two phases**: **prefill** processes the prompt in a single pass and builds the KV cache; **decode** feeds only the new token at each step and reads the historical KV. Prefill for long prompts is usually more compute-bound, while decode at low batch is often bound by the bandwidth of reading weights/KV; the KV cache eliminates the redundant computation of historical tokens, but it does not eliminate the reads of the historical KV. Service evaluation should look together at **TTFT** (first-token latency, including queuing and prefill), **TPOT** (the time per subsequent token), throughput, and peak GPU memory, and the optimization target depends on request length and concurrency.

| Technical direction | Mainstream approaches and key points |
| :-- | :-- |
| Request scheduling | **Continuous batching** adds/removes requests at every generation step; **chunked prefill** splits a long prompt into chunks to reduce its blocking of decode. Large-scale serving can also disaggregate prefill/decode, but must bear the cost of KV transfer. |
| KV management | **Paged KV / PagedAttention** allocates the cache in blocks to reduce reservation and fragmentation; **prefix caching** reuses the KV of identical prefixes, which requires the model, tokens, positions, and compute configuration to match. GQA/MLA instead shrink the cache from the model-structure side. |
| Operators and runtime | **SDPA / FlashAttention** reduces the intermediate tensors and GPU-memory reads/writes in attention while remaining exact attention; fused operators reduce memory-access/launch overhead. A pre-allocated **static cache** combined with `torch.compile` / **CUDA Graphs** reduces dynamic allocation and scheduling, with the effect depending on shape and hardware. |
| Low precision | BF16/FP16 are common execution precisions; **weight quantization** (INT8/INT4/FP8, etc.) saves weight GPU memory, and **KV quantization** saves the long-context cache. The speedup depends on kernel support, and the precision loss and dequantization overhead must be evaluated. |
| Speculative decoding | **Speculative decoding** has a draft model first propose multiple tokens, which the target model verifies in parallel; correct accept/reject sampling can preserve the target distribution, but not all approximate variants are equivalent. The payoff depends on the acceptance rate, draft overhead, and load. |
| Multi-GPU inference | **TP** partitions the matrices within a layer, **PP** partitions model layers, and MoE's **EP** distributes experts; communication, load balancing, and pipeline bubbles affect the payoff, so you cannot look at the GPU count alone. |

**Generation control must also be rounded out**: repetition / presence / frequency penalties and no-repeat n-gram suppress repetition; **constrained decoding** masks out illegal candidates according to a grammar / JSON schema. Chat models should use the **chat template**, BOS/EOS, and turn-end tokens that match those used during training; multi-request generation must also correctly handle the padding mask, positions, and the per-sequence stopping state. A stop string may span multiple tokens and cannot simply be equated with a single EOS id. Newer options such as min-p and typical sampling can be adopted per task, with no need to implement each one for a basic tutorial.

These industrial-grade capabilities are best learned through frameworks such as [vLLM](https://docs.vllm.ai/en/latest/) and [TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/); for cache types see the [Transformers KV cache documentation](https://huggingface.co/docs/transformers/en/kv_cache), and for the principles of speculative decoding see [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192). This tutorial implements the common sampling methods and the KV cache, and provides a technology map for the rest here.